In [0]:
dbutils.library.restartPython()

In [0]:
# XGBoost Pretzel Forecasting Model — Table of Contents

| Block | Description |
|-------|-------------|
| **Block 1** | Restart, preprocessing, imports, parameters, base prep |
| **Block 2** | Helper functions |
| **Block 3** | Weekly backtest function |
| **Block 4** | Run weeks and materialise predictions |
| **Block 5** | Metrics helpers and eval base |
| **Block 6** | Main metrics |
| **Block 7** | Store performance |
| **Block 8** | Bucket diagnostics |
| **Block 9** | Spike diagnostics |
| **Block 10** | Feature lists |
| **Block 11** | Feature importance |
| **Block 12** | Weekly summary |
| **Block 13** | Baseline function |
| **Block 14** | Baseline runs |
| **Block 15** | Baseline long + eval |
| **Block 16** | Comparison tables |
| **Block 17** | Remaining diagnostics |
| **Block 18** | Signed error distribution and exact / within-k |

---

In [0]:
%run "/Shared/Capstone Project_AA_2026/Data Preprocessing"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# if needed on serverless
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Block 1 — restart, preprocessing, imports, parameters, base prep

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
from datetime import timedelta
import gc
import numpy as np
import pandas as pd
import xgboost as xgb

# =============================================================================
# MODEL & BEHAVIOUR PARAMETERS
# =============================================================================
# These are the knobs you'd turn if you wanted to retune the model.
# They were set during model development after testing many combinations —
# most people won't need to touch them.

# --- XGBoost tree settings ---
# XGBoost works by building a sequence of decision trees, each one correcting
# the mistakes of the last. These settings control how complex those trees are.
XGB_N_ESTIMATORS = 350       # How many trees to build
XGB_MAX_DEPTH = 5            # How deep each tree can go (deeper = more complex patterns)
XGB_LEARNING_RATE = 0.04     # How much each new tree contributes — lower = more conservative
XGB_MIN_CHILD_WEIGHT = 10    # Minimum data needed in a leaf — prevents overfitting on tiny groups
XGB_SUBSAMPLE = 0.8          # Each tree only sees 80% of rows — adds randomness, helps generalisation
XGB_COLSAMPLE_BYTREE = 0.75  # Each tree only sees 75% of features — same idea
XGB_REG_LAMBDA = 2.5         # Penalty for complexity — stops the model getting too "wiggly"
XGB_REG_ALPHA = 0.0          # A second type of complexity penalty (unused here)
XGB_TREE_METHOD = "hist"     # Fast histogram-based tree building — works well on large datasets
XGB_RANDOM_STATE = 42        # Fixed seed so results are reproducible run-to-run

# --- Column name aliases ---
ACT_P = "Sales of Pretzels"
ACT_N = "Sales of Pretzel Nuggets"

# --- Data range guard ---
# Rows beyond this date are treated as test/placeholder and filtered out
MAX_ALLOWED_DATE = "2027-12-31"

# --- Zero-demand suppression thresholds ---
# Some store/hour slots basically never sell anything. If the model predicts a
# tiny positive number for one of those, we snap it to zero rather than
# telling the bakery to make e.g. 0.1 pretzels.
# A slot is zeroed only if ALL THREE conditions are true:
#   1. It sold nothing in fewer than 10% (or 8%) of its recent lag history
#   2. The baseline (recent average) is already near zero
#   3. The raw model prediction is also near zero
PRETZEL_ZERO_RATE_THRESHOLD = 0.10
NUGGET_ZERO_RATE_THRESHOLD = 0.08
PRETZEL_BASELINE_ZERO_THRESHOLD = 0.25
NUGGET_BASELINE_ZERO_THRESHOLD = 0.20
PRETZEL_RAW_ZERO_CUTOFF = 0.35
NUGGET_RAW_ZERO_CUTOFF = 0.45

# --- Residual shrinkage ---
# The model predicts the *gap* between the baseline and actual sales (the residual).
# We don't trust that gap 100% — we shrink it toward zero so the model
# leans on the baseline more and doesn't overcorrect.
# 0.5 means we apply 50% of the model's suggested correction for pretzels.
PRETZEL_RESIDUAL_SHRINK = 0.50
NUGGET_RESIDUAL_SHRINK = 0.35

# --- Calibration clamp ---
# After training we check if the model is systematically biased for certain
# time slots (e.g. always predicting too high on weekend evenings).
# We apply a multiplier to fix that, but only allow it to go downward —
# the model tends to over-predict, not under-predict.
# 0.92 = we'll correct by at most 8% downward per slot.
PRETZEL_CALIBRATION_MIN = 0.92
PRETZEL_CALIBRATION_MAX = 1.00
NUGGET_CALIBRATION_MIN = 0.92
NUGGET_CALIBRATION_MAX = 1.00

# --- Training weight settings (currently uniform) ---
# These would let you make the model pay more attention to hours with actual sales
# vs. zero-sales hours. Currently set to treat all rows equally.
PRETZEL_POSITIVE_ROW_WEIGHT = 1.0
NUGGET_POSITIVE_ROW_WEIGHT = 1.0
PRETZEL_VOLUME_WEIGHT_POWER = 0.0
NUGGET_VOLUME_WEIGHT_POWER = 0.0
MAX_TRAIN_WEIGHT = 1.0

# --- Minimum rows needed to fit a reliable model ---
MIN_SEGMENT_TRAIN_ROWS = 4000
MIN_SEGMENT_TEST_ROWS = 1

# --- Stores excluded from training ---
# These stores have known data quality issues (missing periods, anomalous patterns)
# that would confuse the model if included. They're still scored — we just don't
# let their history influence what the model learns.
TRAIN_EXCLUDED_STORES = [
    "40653fdaa1a7b099f8e2940e17f9de646ae205874a260b76d67d6ca2e389bb0b",
    "5314ac3a1ced59cac780e91d41863fa3864bf795d4e3bd650926baaaa05ff9dd",
    "f3b09748ccbbf8ae083e25dd0c1efdef211a35944238f47100caa96ca2d3ad4f",
    "53f5305ed859a788ae27ccc150b84309c704000bb0dd44c1e5024a91e37c15c9",
    "cb7a43d9cea176829ae0f790924a55441a6631f95946b628032db210f0abbf34",
    "03abf2813ae64f3d4c6d15f7eec8d2fa45fd74abc4d0fb08239ddad9e7520094",
    "ac3bf16e15a76b006e569bca767d49a680637b6f05947647146516aebefbc5da",
    "3fce676db6b8ae6c0ccaa27c16ca246d586d50642a7ed53a6740252b83f46564",
    "c3c1ab6522e89dcbd5e0d86f20ed1865055f68c13a44497074ee599914a735da",
    "494d4f7222fc7d2d4b3a328b4a1a4311f7788d950d064806bc4be38d82f708f3",
    "3144d7983c8351ba133e0a7a817a23d40c58f893bdc750d851bede97149e340b",
    "29ca1d89c65f5f70b5e20ccd68fc9e9d18fb477d41c26810d3f0c18a3cd1b12f",
    "c94b0e865aee0f73db9b98ef50cfba1a09b2ee5d8db8f2c58b6ff29911815aa2",
    "659a96b5ec100bbd2066acc8b9cc690e33daafcb4a8e3234e96db0cb92d86b1c",
    "7b40adf51bb6a75e5971a755191ad5707993e56e143d8f630ae2cb72883639a2",
    "747a1a73ac9cef4b1985a4949af423c4fd91fbe9da69ac72ac0c1133a3c992f5",
    "55be0a5d2b10f3be9327c8ba51006e6cef0c75aa7c35608de34fefb1c7db5fb0",
    "8c584d33a2c897d8975f82409aa6948d9c3ce56a3176e28af60df1518b347522",
    "32aaefdbdae31bba3ebf9516d6459d1686660405cb5858e724fc7b959fe85d80",
    "36358123b673fbdd7f7a71f7c64684c33c4e64b8765fca6fe65e581f248ebfc6",
    "17ab67a6429b61be62bd717aa8aedee5649684da0ae6c7a5de2a889237f7b31c",
    "a888204079972c6691043ca05ada6c4fa27c5d4041317ca958981c204af55525",
    "5914ac87d84fbe24c972b10804c35e033d740aeff69fc9821b1cc2557144b9b0",
    "8b99338c2e8ec16cb2540e876d1eb28649776121ecb3b4353d79626f6f67814d",
    "61954112f62fecf525964b4b793b4211ed9fe6f18a0db217158cdfbe34b606a8",
    "a31f22400e17eb1c948e4f417bf7d56cc5095980b3ccfddad98523f8c5e6c736",
    "38d91e3f2e38ef928a1f9d8e0db0e4032a300ddbc78d5af688aa3d6332be937f",
    "0178eb1a08c88a741fb58a649a4bbc1d84e0e122c681ca19a049b14fad810a43",
    "83d9a11e4b0b31b981d8502baf015ef15492335c6ab750d6c72c37cfa1c7a6e2",
    "78e272ecc6d53bd87be4e303b3d6dd9c9d0874b4b8e3b1c263f1a29abff3a470",
    "a5ec24c5bbba98b541db5062036665ed318aa54113a8b649a738faa35a2c061c",
    "443706408975580b430b1a5d0bf25200dbf303e55d0a1724f2e2569332f9008f",
    "be3959a472acea8f3afa0cba5346aed1e3bac12bef34866c20bb9ab07adf7fdf",
    "f3ee75f6fb2ea19b21f08a09b29b930bdb799f07b56696ba68e92609d47cf969",
    "bbf62751c1c819da932154675866c668885253fe80de4d7e6adf109f1e62d7d5",
    "f09c5d013b033d6d227a95f2129f980a178fd7328a97551a28c20b717db30726",
    "b90493ade251b461dc13ecf0f311ea02a6855a539798895d79060ea59b12c62b",
    "47b9e2c57a19261f205d4c33a2c8c477114d71a6e25d68139cdc323eba186902",
    "8a8a1e51c304cff8461a4509b2062e1aa2dc2dbb295e05bf0cce0ef445cbd5c7",
    "b460da8436d125e7339a437faa83631b7a5c74f0f47c59d11b8f342b2a6276d2",
    "7d9a7deec9c3d15cf02684c78309127a97b900a22db3361a1dd223f11150a33b",
    "77e43af9f855a869e4e62614cb899447fd0e46ebbe06a85a78f1d30d301fca2d",
    "95e582dacca4924b4ca99ec2ac824aff954efa54bfeeb12faaa48ff1d7eed7ef",
    "b05e7a1e2842fc108c639f975f708e3998605f7639218f667528ada27484500b",
    "44e55e6f24c8cf36b1cc91f96e3042851aa52f20f9cbc731db2c8008898852e8",
    "fbb1281b5147d895b77d230bdcb23fc29ada3b4f2c4f1afa3b0d202ef92b4f07",
    "d790b84b95e7a9e6d44b2d761c3fd4e62e20dbe63a7043372d704725099a9d2d",
    "a2eb262f79d05b81d00ede44c400dec05380a755da4b9c776004732fb2e887cc",
    "f7ba9d784be32a9a3fef82aafd6fd45cb70c9acb272f2f2e4e9a61ce697869d8",
    "a4c8a3ce718fbb2adaf7e71d54bbf941ed5854d0e25f77ea5ce396be9476dbf0"
]

# =============================================================================
# DATA PREPARATION
# =============================================================================
# Start from the raw hourly_sales table and apply filters and transformations
# to get it into a clean, usable shape for model training and evaluation.

full = (
    hourly_sales
    # Parse the timestamp string into an actual datetime for date arithmetic
    .withColumn("Time by hour", F.to_timestamp("Time by hour", "yyyy-MM-dd HH:mm"))
    # Drop dummy rows from year 2031 — these are placeholders in the source data
    .filter(F.year("Time by hour") != 2031)
    # Only keep trading hours (10am–10pm). Outside this window stores aren't open.
    .filter(F.col("hour").between(10, 22))
    # Don't include any data beyond our allowed horizon
    .filter(F.to_date("Time by hour") <= F.lit(MAX_ALLOWED_DATE))
    # Flag opening and closing hours — these have distinct sales patterns
    # (slow start at open, predictable ramp-down near close)
    .withColumn("is_opening_hour", F.when(F.col("hour") == 10, 1).otherwise(0))
    .withColumn("is_closing_hour", F.when(F.col("hour").isin(20, 21, 22), 1).otherwise(0))
    # Assign each row to a Mon–Sun forecast week. This is the core unit we train
    # and evaluate on — everything is sliced by week boundary.
    .withColumn(
        "forecast_week_start_date",
        F.date_sub(F.next_day(F.to_date("Time by hour"), "Mon"), 7)
    )
    .withColumn("forecast_week_start", F.to_timestamp("forecast_week_start_date"))
)

# Materialise the dataset — this forces Spark to actually read the data once
# rather than re-reading it from scratch on every subsequent operation
_ = full.count()

# Find the latest timestamp in the data to determine how far history goes
max_ts = full.agg(F.max("Time by hour").alias("m")).collect()[0]["m"]
max_date = max_ts.date()

# Only treat today as fully complete once the last trading hour (10pm) has landed.
# Running at e.g. 5pm on a Sunday would see Sunday's date but have incomplete data —
# this guard prevents that week from being selected as eval.
if max_ts.hour < 22:
    effective_max_date = max_date - timedelta(days=1)
else:
    effective_max_date = max_date

# Find all weeks where we have a full Mon–Sun of data.
# A week is complete when its Sunday falls on or before our effective cutoff.
complete_weeks = (
    full
    .select("forecast_week_start", "forecast_week_start_date")
    .distinct()
    .withColumn("forecast_week_end_date", F.date_add("forecast_week_start_date", 6))
    .filter(F.col("forecast_week_end_date") <= F.lit(effective_max_date))
    .orderBy("forecast_week_start")
    .collect()
)

# Take the two most recent complete weeks as the eval window.
# The model trains on everything before the first of these — clean temporal split,
# no leakage of future data into training.
complete_weeks = complete_weeks[-2:]

eval_week_starts = [r["forecast_week_start"] for r in complete_weeks]

print("max timestamp:", max_ts)
print("evaluation weeks:")
for ws in eval_week_starts:
    print(ws)


max timestamp: 2026-04-20 22:00:00
evaluation weeks:
2026-04-06 00:00:00
2026-04-13 00:00:00


features added successfully!
total columns: 94

column names:
  - week_start_date
  - Store id
  - business_date
  - hour
  - venue
  - dma
  - city
  - state
  - brand
  - Sales of Pretzels
  - Sales of Pretzel Nuggets
  - Time by hour
  - day_of_week
  - is_weekend
  - is_holiday
  - is_holiday_or_holiday_week
  - hour_sin
  - hour_cos
  - Lag_1_week_Pretzels
  - Lag_1_week_Nuggets
  - Lag_2_week_Pretzels
  - Lag_2_week_Nuggets
  - Lag_3_week_Pretzels
  - Lag_3_week_Nuggets
  - Lag_4_week_Pretzels
  - Lag_4_week_Nuggets
  - SameSlot_2week_avg_Pretzels
  - SameSlot_2week_avg_Nuggets
  - SameSlot_2week_std_Pretzels
  - SameSlot_2week_std_Nuggets
  - SameSlot_2week_min_Pretzels
  - SameSlot_2week_min_Nuggets
  - SameSlot_2week_max_Pretzels
  - SameSlot_2week_max_Nuggets
  - SameSlot_3week_avg_Pretzels
  - SameSlot_3week_avg_Nuggets
  - SameSlot_3week_std_Pretzels
  - SameSlot_3week_std_Nuggets
  - SameSlot_3week_min_Pretzels
  - SameSlot_3week_min_Nuggets
  - SameSlot_3week_max_Pretzels

week_start_date,Store id,business_date,hour,venue,dma,city,state,brand,Sales of Pretzels,Sales of Pretzel Nuggets,Time by hour,day_of_week,is_weekend,is_holiday,is_holiday_or_holiday_week,hour_sin,hour_cos,Lag_1_week_Pretzels,Lag_1_week_Nuggets,Lag_2_week_Pretzels,Lag_2_week_Nuggets,Lag_3_week_Pretzels,Lag_3_week_Nuggets,Lag_4_week_Pretzels,Lag_4_week_Nuggets,SameSlot_2week_avg_Pretzels,SameSlot_2week_avg_Nuggets,SameSlot_2week_std_Pretzels,SameSlot_2week_std_Nuggets,SameSlot_2week_min_Pretzels,SameSlot_2week_min_Nuggets,SameSlot_2week_max_Pretzels,SameSlot_2week_max_Nuggets,SameSlot_3week_avg_Pretzels,SameSlot_3week_avg_Nuggets,SameSlot_3week_std_Pretzels,SameSlot_3week_std_Nuggets,SameSlot_3week_min_Pretzels,SameSlot_3week_min_Nuggets,SameSlot_3week_max_Pretzels,SameSlot_3week_max_Nuggets,SameSlot_4week_avg_Pretzels,SameSlot_4week_avg_Nuggets,SameSlot_4week_std_Pretzels,SameSlot_4week_std_Nuggets,SameSlot_4week_min_Pretzels,SameSlot_4week_min_Nuggets,SameSlot_4week_max_Pretzels,SameSlot_4week_max_Nuggets,PrevWeek_AllHourAvg_Pretzels,PrevWeek_AllHourStd_Pretzels,PrevWeek_AllHourMin_Pretzels,PrevWeek_AllHourMax_Pretzels,PrevWeek_AllHourAvg_Nuggets,PrevWeek_AllHourStd_Nuggets,PrevWeek_AllHourMin_Nuggets,PrevWeek_AllHourMax_Nuggets,Prev2Week_AllHourAvg_Pretzels,Prev2Week_AllHourStd_Pretzels,Prev2Week_AllHourMin_Pretzels,Prev2Week_AllHourMax_Pretzels,Prev2Week_AllHourAvg_Nuggets,Prev2Week_AllHourStd_Nuggets,Prev2Week_AllHourMin_Nuggets,Prev2Week_AllHourMax_Nuggets,Prev3Week_AllHourAvg_Pretzels,Prev3Week_AllHourStd_Pretzels,Prev3Week_AllHourMin_Pretzels,Prev3Week_AllHourMax_Pretzels,Prev3Week_AllHourAvg_Nuggets,Prev3Week_AllHourStd_Nuggets,Prev3Week_AllHourMin_Nuggets,Prev3Week_AllHourMax_Nuggets,Prev4Week_AllHourAvg_Pretzels,Prev4Week_AllHourStd_Pretzels,Prev4Week_AllHourMin_Pretzels,Prev4Week_AllHourMax_Pretzels,Prev4Week_AllHourAvg_Nuggets,Prev4Week_AllHourStd_Nuggets,Prev4Week_AllHourMin_Nuggets,Prev4Week_AllHourMax_Nuggets,AllHourAvg_2week_mean_Pretzels,AllHourAvg_2week_mean_Nuggets,AllHourAvg_3week_mean_Pretzels,AllHourAvg_3week_mean_Nuggets,AllHourAvg_4week_mean_Pretzels,AllHourAvg_4week_mean_Nuggets,AllHourStd_2week_mean_Pretzels,AllHourStd_2week_mean_Nuggets,AllHourStd_3week_mean_Pretzels,AllHourStd_3week_mean_Nuggets,AllHourStd_4week_mean_Pretzels,AllHourStd_4week_mean_Nuggets
2024-01-01,0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2024-01-02,10,Hypermart,Salisbury MD,Milford,DE,Auntie Anne's,8.0,1.0,2024-01-02T10:00:00.000Z,3,0,0,0,0.49999999999999994,-0.8660254037844387,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2024-01-01,0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2024-01-02,11,Hypermart,Salisbury MD,Milford,DE,Auntie Anne's,13.0,3.0,2024-01-02T11:00:00.000Z,3,0,0,0,0.258819045102521,-0.9659258262890682,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2024-01-01,0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2024-01-02,12,Hypermart,Salisbury MD,Milford,DE,Auntie Anne's,18.0,4.0,2024-01-02T12:00:00.000Z,3,0,0,0,1.2246467991473532E-16,-1.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,n

transaction_order_line rows: 308655439
transaction_order rows: 156943653
menu_raw rows: 7983
menu final rows: 7825


barcode,count


...

+---------+------------------+---------------+-------------+--------------+---------------+
|     rows|distinct_store_ids|venue_null_rows|dma_null_rows|city_null_rows|state_null_rows|
+---------+------------------+---------------+-------------+--------------+---------------+
|308515880|              1115|         299321|       261229|        748929|         748929|
+---------+------------------+---------------+-------------+--------------+---------------+



+----------------------+---------+
|venue_status          |count    |
+----------------------+---------+
|Mall - Interior Retail|191640381|
|Mall - Outlets        |43820895 |
|Airport               |23113835 |
|Hypermart             |9299927  |
|Retail Street         |6584244  |
|Strip                 |5125868  |
|Mall - Open Air       |4945836  |
|Transit Hub           |4468058  |
|Lifestyle Center      |2976121  |
|Food Truck            |2931008  |
|Power Center          |2637359  |
|Strip/Grocery Anch.   |2187106  |
|Military Base         |1091618  |
|Grocery               |1000704  |
|Boardwalk             |950676   |
|Stand-alone           |792702   |
|Hotel                 |764929   |
|Theme Park            |739063   |
|Casino                |551115   |
|Department Store      |453374   |
|Travel Plaza          |415067   |
|Zoo                   |371188   |
|Mall - Exterior       |311747   |
|NULL                  |299321   |
|Travel Center         |292058   |
|Mall               

+-------------------+----+------------------+
|current_record_flag|rows|distinct_store_ids|
+-------------------+----+------------------+
|               NULL|7038|              7038|
|                  N|  96|                96|
|                  Y|  53|                53|
+-------------------+----+------------------+



all dim distinct stores: 7187


+----------------------+--------+
|venue                 |count   |
+----------------------+--------+
|Mall - Interior Retail|91971664|
|Mall - Outlets        |20260550|
|Hypermart             |4579914 |
|Mall - Open Air       |2221249 |
|Retail Street         |2147145 |
|Transit Hub           |2089891 |
|Strip                 |1700518 |
|Food Truck            |1397400 |
|Lifestyle Center      |1281690 |
|Power Center          |1170277 |
|Strip/Grocery Anch.   |738696  |
|Grocery               |520479  |
|Military Base         |496459  |
|Boardwalk             |407445  |
|Casino                |278581  |
|Theme Park            |233366  |
|Stand-alone           |227518  |
|Travel Plaza          |202894  |
|Department Store      |166220  |
|Mall - Exterior       |165936  |
|Hotel                 |143150  |
|Mall                  |132939  |
|Zoo                   |128131  |
|Travel Center         |113128  |
|Farmers Market        |100952  |
|Hospital              |52630   |
|College/Unive

Block 2 — helper functions

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from datetime import timedelta
import gc
import numpy as np
import pandas as pd
import xgboost as xgb

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def safe_fill_zero(df, cols):
    # Replace nulls with 0 in numeric feature columns.
    # Nulls at this stage mean "no sales in that lag window" — zero is the right default.
    existing = [c for c in cols if c in df.columns]
    for c in existing:
        df = df.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
    return df

def add_nonzero_rate(df, lag_cols, out_col):
    # For each row, calculate what fraction of the last 4 same-slot weeks had any sales at all.
    # e.g. if pretzels sold in 3 of the last 4 same-day/same-hour slots, this returns 0.75.
    # This is one of the strongest signals for zero suppression — a slot that's almost
    # always zero should probably be predicted as zero regardless of what the model says.
    available = [c for c in lag_cols if c in df.columns]
    if not available:
        return df.withColumn(out_col, F.lit(0.0))

    nonzero_expr = None
    valid_expr = None

    for c in available:
        nz = F.when(F.col(c).isNotNull(), F.when(F.col(c) > 0, 1.0).otherwise(0.0)).otherwise(0.0)
        vd = F.when(F.col(c).isNotNull(), 1.0).otherwise(0.0)
        nonzero_expr = nz if nonzero_expr is None else nonzero_expr + nz
        valid_expr = vd if valid_expr is None else valid_expr + vd

    return df.withColumn(
        out_col,
        F.when(valid_expr > 0, nonzero_expr / valid_expr).otherwise(0.0)
    )

def build_pretzel_baseline_expr():
    # The baseline is our best simple estimate before the ML model gets involved.
    # It's a priority-ordered coalesce: use the 4-week same-slot average if available,
    # fall back to 3-week, 2-week, last week's value, then progressively broader
    # store-level averages. Every row ends up with some baseline — even brand new stores.
    return F.coalesce(
        F.col("SameSlot_4week_avg_Pretzels"),
        F.col("SameSlot_3week_avg_Pretzels"),
        F.col("SameSlot_2week_avg_Pretzels"),
        F.col("Lag_1_week_Pretzels"),
        F.col("avg_sales_store_hour_dow_pretzels"),
        F.col("avg_sales_store_hour_pretzels"),
        F.col("avg_sales_store_dow_pretzels"),
        F.col("avg_sales_store_pretzels"),
        F.lit(0.0)
    )

def build_nugget_baseline_expr():
    return F.coalesce(
        F.col("SameSlot_4week_avg_Nuggets"),
        F.col("SameSlot_3week_avg_Nuggets"),
        F.col("SameSlot_2week_avg_Nuggets"),
        F.col("Lag_1_week_Nuggets"),
        F.col("avg_sales_store_hour_dow_nuggets"),
        F.col("avg_sales_store_hour_nuggets"),
        F.col("avg_sales_store_dow_nuggets"),
        F.col("avg_sales_store_nuggets"),
        F.lit(0.0)
    )

def add_delta_features(df):
    # Give the model information about how much the baseline differs from last week.
    # If the baseline is much higher than last week, something changed (upcoming holiday,
    # seasonal shift). The model can decide whether to trust or discount that gap.
    if "Lag_1_week_Pretzels" in df.columns:
        df = df.withColumn(
            "pretzel_baseline_minus_lag1",
            F.col("baseline_p") - F.coalesce(F.col("Lag_1_week_Pretzels"), F.lit(0.0))
        )
        df = df.withColumn(
            "pretzel_baseline_to_lag1_ratio",
            F.when(F.col("Lag_1_week_Pretzels") > 0, F.col("baseline_p") / F.col("Lag_1_week_Pretzels")).otherwise(1.0)
        )
    else:
        df = df.withColumn("pretzel_baseline_minus_lag1", F.lit(0.0))
        df = df.withColumn("pretzel_baseline_to_lag1_ratio", F.lit(1.0))

    # Recent volatility: how much does this slot vary week-to-week?
    # High-volatility slots are harder to predict — the model can factor this in.
    if "SameSlot_4week_std_Pretzels" in df.columns:
        df = df.withColumn(
            "pretzel_recent_volatility",
            F.coalesce(F.col("SameSlot_4week_std_Pretzels"), F.lit(0.0))
        )
    else:
        df = df.withColumn("pretzel_recent_volatility", F.lit(0.0))

    if "Lag_1_week_Nuggets" in df.columns:
        df = df.withColumn(
            "nugget_baseline_minus_lag1",
            F.col("baseline_n") - F.coalesce(F.col("Lag_1_week_Nuggets"), F.lit(0.0))
        )
        df = df.withColumn(
            "nugget_baseline_to_lag1_ratio",
            F.when(F.col("Lag_1_week_Nuggets") > 0, F.col("baseline_n") / F.col("Lag_1_week_Nuggets")).otherwise(1.0)
        )
    else:
        df = df.withColumn("nugget_baseline_minus_lag1", F.lit(0.0))
        df = df.withColumn("nugget_baseline_to_lag1_ratio", F.lit(1.0))

    if "SameSlot_4week_std_Nuggets" in df.columns:
        df = df.withColumn(
            "nugget_recent_volatility",
            F.coalesce(F.col("SameSlot_4week_std_Nuggets"), F.lit(0.0))
        )
    else:
        df = df.withColumn("nugget_recent_volatility", F.lit(0.0))

    return df

def build_segment_calibration_pdf(train_pdf, actual_col, pred_col, out_col, min_ratio, max_ratio):
    # After training, we check: grouped by hour + weekday/weekend, what is the ratio
    # of actual total sales to predicted total sales on the training data?
    # If the model predicted 1000 but actual was 920, multiplier = 0.92 — we systematically
    # over-predict this slot type and should scale down.
    # The min/max clamp stops us making overcorrections.
    calib = (
        train_pdf
        .groupby(["hour", "is_weekend"], as_index=False)
        .agg(actual_sum=(actual_col, "sum"), pred_sum=(pred_col, "sum"))
    )
    calib[out_col] = np.where(calib["pred_sum"] > 0, calib["actual_sum"] / calib["pred_sum"], 1.0)
    calib[out_col] = calib[out_col].clip(lower=min_ratio, upper=max_ratio)
    return calib[["hour", "is_weekend", out_col]]

def make_train_weights(pdf, actual_col, positive_row_weight, volume_weight_power, max_train_weight):
    # Optionally up-weight rows where sales actually happened, or weight by volume.
    # Currently all parameters are set so this returns uniform weights of 1.0.
    # Kept here in case we want to experiment with weighted training later.
    actual = pdf[actual_col].to_numpy(dtype=np.float32)
    positive_flag = (actual > 0).astype(np.float32)
    positive_component = 1.0 + positive_flag * (positive_row_weight - 1.0)
    volume_component = np.where(
        actual > 0,
        np.power(actual + 1.0, volume_weight_power),
        1.0
    ).astype(np.float32)
    weights = positive_component * volume_component
    weights = np.minimum(weights, max_train_weight).astype(np.float32)
    return weights

def fit_xgb_residual_model(train_pdf, feature_cols, target_col, sample_weight):
    # Train the XGBoost model. The target is the residual (log-scale gap between
    # the baseline and actuals), not raw sales. Working in log-space means a ×2
    # error on a small number and a ×2 error on a large number are treated equally,
    # which is the right behaviour for sales forecasting.
    X_train = train_pdf[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_pdf[target_col].to_numpy(dtype=np.float32)

    model = xgb.XGBRegressor(
        n_estimators=XGB_N_ESTIMATORS,
        max_depth=XGB_MAX_DEPTH,
        learning_rate=XGB_LEARNING_RATE,
        min_child_weight=XGB_MIN_CHILD_WEIGHT,
        subsample=XGB_SUBSAMPLE,
        colsample_bytree=XGB_COLSAMPLE_BYTREE,
        reg_alpha=XGB_REG_ALPHA,
        reg_lambda=XGB_REG_LAMBDA,
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method=XGB_TREE_METHOD,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=1
    )

    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model

def spark_to_pandas(df, cols):
    # Pull only the columns we need from Spark into a local pandas DataFrame.
    # XGBoost runs on a single machine, not distributed — this is the handoff point.
    # For datasets this size it fits in memory fine.
    use_cols = [c for c in cols if c in df.columns]
    return df.select(*use_cols).toPandas()

def create_store_volume_segments(train_df):
    # Divide stores into low / medium / high volume thirds based on average hourly sales.
    # This gives the model a signal about store "size" — a high-volume store and a
    # low-volume store behave differently even at the same hour on the same day of week.
    store_volume = (
        train_df
        .groupBy("Store id")
        .agg(
            F.mean(ACT_P).alias("avg_sales_store_pretzels"),
            F.mean(ACT_N).alias("avg_sales_store_nuggets")
        )
        .withColumn(
            "avg_sales_store_total",
            F.col("avg_sales_store_pretzels") + F.col("avg_sales_store_nuggets")
        )
    )

    q1, q2 = store_volume.approxQuantile("avg_sales_store_total", [1.0 / 3.0, 2.0 / 3.0], 0.001)

    store_volume = (
        store_volume
        .withColumn(
            "store_volume_segment",
            F.when(F.col("avg_sales_store_total") <= F.lit(q1), F.lit("low"))
             .when(F.col("avg_sales_store_total") <= F.lit(q2), F.lit("medium"))
             .otherwise(F.lit("high"))
        )
        .withColumn(
            "store_volume_segment_code",
            F.when(F.col("store_volume_segment") == "low", F.lit(0.0))
             .when(F.col("store_volume_segment") == "medium", F.lit(1.0))
             .otherwise(F.lit(2.0))
        )
    )

    return store_volume, q1, q2

def compare_total_vs_distinct(df, key_cols, label="df"):
    # Sanity check: are there duplicate rows for the same store/timestamp?
    # Duplicates would inflate metrics and cause silent errors downstream.
    total_rows = df.count()
    distinct_rows = df.select(*key_cols).distinct().count()
    duplicate_rows = total_rows - distinct_rows

    result = spark.createDataFrame(
        [(label, total_rows, distinct_rows, duplicate_rows,
          duplicate_rows / total_rows if total_rows > 0 else 0.0)],
        schema=["dataset", "total_rows", "distinct_rows", "duplicate_rows", "duplicate_rate"]
    )

    display(result)
    return result

def mae_zero_vs_positive(eval_df):
    # Split error metrics into two buckets: hours where actual sales were zero vs. positive.
    # This matters because the model's job is different in each case:
    #   - For zero-actual rows, any positive prediction is a false alarm (wasted baking)
    #   - For positive-actual rows, missing the demand means running out (lost sales)
    # Seeing these separately shows which failure mode is more common.
    pretzel_zero_vs_positive = (
        eval_df
        .withColumn(
            "actual_bucket",
            F.when(F.col("Sales of Pretzels") == 0, F.lit("actual = 0"))
             .otherwise(F.lit("actual > 0"))
        )
        .groupBy("actual_bucket")
        .agg(
            F.count("*").alias("N"),
            F.mean("ae_p").alias("MAE_pretzels"),
            F.sum("ae_p").alias("sum_ae_p"),
            F.sum(F.col("Sales of Pretzels")).alias("pretzel_volume")
        )
        .withColumn("metric", F.lit("pretzels"))
        .select("metric", "actual_bucket", "N", "MAE_pretzels", "sum_ae_p", "pretzel_volume")
    )

    nugget_zero_vs_positive = (
        eval_df
        .withColumn(
            "actual_bucket",
            F.when(F.col("Sales of Pretzel Nuggets") == 0, F.lit("actual = 0"))
             .otherwise(F.lit("actual > 0"))
        )
        .groupBy("actual_bucket")
        .agg(
            F.count("*").alias("N"),
            F.mean("ae_n").alias("MAE_nuggets"),
            F.sum("ae_n").alias("sum_ae_n"),
            F.sum(F.col("Sales of Pretzel Nuggets")).alias("nugget_volume")
        )
        .withColumn("metric", F.lit("nuggets"))
        .select("metric", "actual_bucket", "N", "MAE_nuggets", "sum_ae_n", "nugget_volume")
    )

    total_zero_vs_positive = (
        eval_df
        .withColumn(
            "actual_bucket",
            F.when(F.col("actual_total") == 0, F.lit("actual = 0"))
             .otherwise(F.lit("actual > 0"))
        )
        .groupBy("actual_bucket")
        .agg(
            F.count("*").alias("N"),
            F.mean("ae_total").alias("MAE_total"),
            F.sum("ae_total").alias("sum_ae_total"),
            F.sum("actual_total").alias("total_volume")
        )
        .withColumn("metric", F.lit("total"))
        .select("metric", "actual_bucket", "N", "MAE_total", "sum_ae_total", "total_volume")
    )

    return pretzel_zero_vs_positive, nugget_zero_vs_positive, total_zero_vs_positive


Block 3 — weekly backtest function

In [0]:
# COMMAND ----------

# =============================================================================
# MAIN TRAINING AND SCORING FUNCTION
# =============================================================================
# This is the core of the notebook. It trains one global XGBoost model on all
# history before the eval window, then scores every store/hour slot in those weeks.
# "Global" means one model across all stores — this gives the model far more data
# to learn from than training separate per-store models would.

def run_single_train_two_week_score(full_df, eval_week_starts):

    first_eval_week = eval_week_starts[0]
    last_eval_week = eval_week_starts[-1]
    test_end = last_eval_week + timedelta(days=7)

    print("training cutoff:", first_eval_week)
    print("scoring window:", first_eval_week, "to", test_end)
    print("training-only excluded stores:", len(TRAIN_EXCLUDED_STORES))

    # Remove stores with known data quality issues from both train and test.
    # They're still scored — we just don't let their bad history teach the model anything.
    eligible_df = full_df.filter(~F.col("Store id").isin(TRAIN_EXCLUDED_STORES))

    # Clean temporal split — the model never sees data from the weeks it's predicting.
    # Training = everything before the first eval week.
    # Test = exactly the two eval weeks.
    train_df = eligible_df.filter(F.col("forecast_week_start") < F.lit(first_eval_week))

    test_df = eligible_df.filter(
        (F.col("forecast_week_start").isin(eval_week_starts)) &
        (F.col("Time by hour") >= F.lit(first_eval_week)) &
        (F.col("Time by hour") < F.lit(test_end))
    )

    print("train rows after store exclusion:", train_df.count())
    print("test rows retained:", test_df.count())

    # Cap extreme outlier sales at the 99.5th percentile of training data.
    # A store that once sold 500 pretzels in an hour would otherwise pull the model
    # toward predicting unrealistic numbers. We apply the same cap to test for consistency.
    p995_pretzels = train_df.approxQuantile(ACT_P, [0.995], 0.001)[0]
    p995_nuggets = train_df.approxQuantile(ACT_N, [0.995], 0.001)[0]

    print("train pretzels 99.5th percentile:", p995_pretzels)
    print("train nuggets 99.5th percentile:", p995_nuggets)

    train_df = (
        train_df
        .withColumn(ACT_P, F.least(F.col(ACT_P), F.lit(p995_pretzels)))
        .withColumn(ACT_N, F.least(F.col(ACT_N), F.lit(p995_nuggets)))
    )

    # Compute store-level averages at multiple granularities from training data only.
    # These become features that give the model context about how busy a store is —
    # in general, at a specific hour, and at a specific hour on a specific day of week.
    store_avg = train_df.groupBy("Store id").agg(
        F.mean(ACT_P).alias("avg_sales_store_pretzels"),
        F.mean(ACT_N).alias("avg_sales_store_nuggets")
    ).withColumn(
        "avg_sales_store_total",
        F.col("avg_sales_store_pretzels") + F.col("avg_sales_store_nuggets")
    )

    store_hour_avg = train_df.groupBy("Store id", "hour").agg(
        F.mean(ACT_P).alias("avg_sales_store_hour_pretzels"),
        F.mean(ACT_N).alias("avg_sales_store_hour_nuggets")
    )

    store_dow_avg = train_df.groupBy("Store id", "day_of_week").agg(
        F.mean(ACT_P).alias("avg_sales_store_dow_pretzels"),
        F.mean(ACT_N).alias("avg_sales_store_dow_nuggets")
    )

    store_hour_dow_avg = train_df.groupBy("Store id", "hour", "day_of_week").agg(
        F.mean(ACT_P).alias("avg_sales_store_hour_dow_pretzels"),
        F.mean(ACT_N).alias("avg_sales_store_hour_dow_nuggets")
    )

    store_volume_segments, q1, q2 = create_store_volume_segments(train_df)
    print("store volume cutoffs:", q1, q2)

    store_volume_segments_join = store_volume_segments.select(
        "Store id", "store_volume_segment", "store_volume_segment_code"
    )

    # Join all store-level context features onto train and test
    train_df = (
        train_df
        .join(store_avg, "Store id", "left")
        .join(store_hour_avg, ["Store id", "hour"], "left")
        .join(store_dow_avg, ["Store id", "day_of_week"], "left")
        .join(store_hour_dow_avg, ["Store id", "hour", "day_of_week"], "left")
        .join(store_volume_segments_join, "Store id", "left")
        .withColumn("store_volume_segment", F.coalesce(F.col("store_volume_segment"), F.lit("medium")))
        .withColumn("store_volume_segment_code", F.coalesce(F.col("store_volume_segment_code"), F.lit(1.0)))
        .withColumn("avg_sales_store_total", F.coalesce(F.col("avg_sales_store_total"), F.lit(0.0)))
    )

    test_df = (
        test_df
        .join(store_avg, "Store id", "left")
        .join(store_hour_avg, ["Store id", "hour"], "left")
        .join(store_dow_avg, ["Store id", "day_of_week"], "left")
        .join(store_hour_dow_avg, ["Store id", "hour", "day_of_week"], "left")
        .join(store_volume_segments_join, "Store id", "left")
        .withColumn("store_volume_segment", F.coalesce(F.col("store_volume_segment"), F.lit("medium")))
        .withColumn("store_volume_segment_code", F.coalesce(F.col("store_volume_segment_code"), F.lit(1.0)))
        .withColumn("avg_sales_store_total", F.coalesce(F.col("avg_sales_store_total"), F.lit(0.0)))
    )

    # Fill nulls with zero — at this point nulls mean "no data for that lag", which is correctly zero
    fill_cols = [
        "avg_sales_store_pretzels", "avg_sales_store_nuggets", "avg_sales_store_total",
        "avg_sales_store_hour_pretzels", "avg_sales_store_hour_nuggets",
        "avg_sales_store_dow_pretzels", "avg_sales_store_dow_nuggets",
        "avg_sales_store_hour_dow_pretzels", "avg_sales_store_hour_dow_nuggets",
        "Lag_1_week_Pretzels", "Lag_2_week_Pretzels", "Lag_3_week_Pretzels", "Lag_4_week_Pretzels",
        "Lag_1_week_Nuggets", "Lag_2_week_Nuggets", "Lag_3_week_Nuggets", "Lag_4_week_Nuggets",
        "SameSlot_4week_std_Pretzels", "SameSlot_4week_std_Nuggets",
        "SameSlot_4week_avg_Pretzels", "SameSlot_4week_avg_Nuggets",
        "SameSlot_3week_avg_Pretzels", "SameSlot_3week_avg_Nuggets",
        "SameSlot_2week_avg_Pretzels", "SameSlot_2week_avg_Nuggets",
        "store_volume_segment_code"
    ]

    train_df = safe_fill_zero(train_df, fill_cols)
    test_df = safe_fill_zero(test_df, fill_cols)

    # Compute how often each store/hour slot actually had sales over the last 4 weeks.
    # This is one of the strongest predictors of whether a slot will be zero or not.
    train_df = add_nonzero_rate(
        train_df,
        ["Lag_1_week_Pretzels", "Lag_2_week_Pretzels", "Lag_3_week_Pretzels", "Lag_4_week_Pretzels"],
        "pretzel_nonzero_rate_4w"
    )
    test_df = add_nonzero_rate(
        test_df,
        ["Lag_1_week_Pretzels", "Lag_2_week_Pretzels", "Lag_3_week_Pretzels", "Lag_4_week_Pretzels"],
        "pretzel_nonzero_rate_4w"
    )
    train_df = add_nonzero_rate(
        train_df,
        ["Lag_1_week_Nuggets", "Lag_2_week_Nuggets", "Lag_3_week_Nuggets", "Lag_4_week_Nuggets"],
        "nugget_nonzero_rate_4w"
    )
    test_df = add_nonzero_rate(
        test_df,
        ["Lag_1_week_Nuggets", "Lag_2_week_Nuggets", "Lag_3_week_Nuggets", "Lag_4_week_Nuggets"],
        "nugget_nonzero_rate_4w"
    )

    # Build the simple history-based baseline — what we'd predict with no ML at all.
    # The XGBoost model's job is to improve on this by predicting the residual gap.
    train_df = (
        train_df
        .withColumn("baseline_p", build_pretzel_baseline_expr())
        .withColumn("baseline_n", build_nugget_baseline_expr())
    )
    test_df = (
        test_df
        .withColumn("baseline_p", build_pretzel_baseline_expr())
        .withColumn("baseline_n", build_nugget_baseline_expr())
    )

    train_df = add_delta_features(train_df)
    test_df = add_delta_features(test_df)

    # Compute the residual target: log(actual + 1) minus log(baseline + 1).
    # This is what XGBoost learns to predict — the log-scale correction to the baseline.
    # Log-space means proportional errors are treated equally regardless of scale.
    train_df = (
        train_df
        .withColumn("pretzel_target_resid", F.log1p(F.col(ACT_P)) - F.log1p(F.col("baseline_p")))
        .withColumn("nugget_target_resid", F.log1p(F.col(ACT_N)) - F.log1p(F.col("baseline_n")))
    )

    # --- Feature sets ---
    # Base features are time/calendar signals shared by both products.
    # Product features add the baseline, lag values, and volatility specific to each product.
    base_features = [
        "day_of_week",                # Mon=0 ... Sun=6
        "is_weekend",                 # binary flag — weekends have distinct patterns
        "is_holiday",                 # binary flag for public holidays
        "is_holiday_or_holiday_week", # broader holiday proximity flag
        "hour_sin",                   # sine encoding of hour (cyclical — 10pm is close to 10am on the circle)
        "hour_cos",                   # cosine encoding of hour
        "is_opening_hour",            # first hour of trading (10am)
        "is_closing_hour"             # final hours of trading (8–10pm)
    ]

    pretzel_features = [
        "baseline_p",                        # simple history-based estimate
        "pretzel_nonzero_rate_4w",           # how often this slot actually sells
        "avg_sales_store_hour_dow_pretzels", # store's average at this exact hour + day combo
        "Lag_1_week_Pretzels",               # what sold exactly 1 week ago in this slot
        "SameSlot_4week_std_Pretzels",       # how variable this slot is week-to-week
        "pretzel_baseline_minus_lag1",       # is the baseline higher or lower than last week?
        "pretzel_recent_volatility"          # same as std above, kept for clarity
    ] + base_features

    nugget_features = [
        "baseline_n",
        "nugget_nonzero_rate_4w",
        "avg_sales_store_hour_dow_nuggets",
        "Lag_1_week_Nuggets",
        "SameSlot_4week_std_Nuggets",
        "nugget_baseline_minus_lag1",
        "nugget_recent_volatility"
    ] + base_features

    # Drop any feature not present in both train and test — can happen for new stores
    pretzel_features = [c for c in pretzel_features if c in train_df.columns and c in test_df.columns]
    nugget_features = [c for c in nugget_features if c in train_df.columns and c in test_df.columns]

    print("pretzel features:", len(pretzel_features))
    print("nugget features:", len(nugget_features))

    # Pull only the columns we need into pandas — avoids moving unnecessary data off the cluster
    train_cols = list(dict.fromkeys(
        ["Store id", "store_volume_segment", "store_volume_segment_code",
         "hour", "is_weekend", ACT_P, ACT_N, "baseline_p", "baseline_n",
         "pretzel_target_resid", "nugget_target_resid"]
        + pretzel_features + nugget_features
    ))

    test_cols = list(dict.fromkeys(
        ["Store id", "Time by hour", "forecast_week_start", "store_volume_segment",
         "store_volume_segment_code", "hour", "day_of_week", "is_weekend", "is_holiday",
         "is_holiday_or_holiday_week", "is_opening_hour", "is_closing_hour",
         ACT_P, ACT_N, "baseline_p", "baseline_n",
         "pretzel_nonzero_rate_4w", "nugget_nonzero_rate_4w"]
        + pretzel_features + nugget_features
    ))

    train_pd = spark_to_pandas(train_df, train_cols)
    test_pd = spark_to_pandas(test_df, test_cols)

    # Drop rows with missing values before training — XGBoost can't handle NaNs.
    # We keep separate DataFrames per product so a missing value for one product
    # doesn't throw away a perfectly valid row for the other.
    train_pd_p = train_pd.dropna(
        subset=pretzel_features + ["pretzel_target_resid", ACT_P, "baseline_p", "hour", "is_weekend"]
    ).copy()
    train_pd_n = train_pd.dropna(
        subset=nugget_features + ["nugget_target_resid", ACT_N, "baseline_n", "hour", "is_weekend"]
    ).copy()

    test_pd_p = test_pd.dropna(
        subset=pretzel_features + [ACT_P, "baseline_p", "pretzel_nonzero_rate_4w", "hour", "is_weekend"]
    ).copy()
    test_pd_n = test_pd.dropna(
        subset=nugget_features + [ACT_N, "baseline_n", "nugget_nonzero_rate_4w", "hour", "is_weekend"]
    ).copy()

    # Train one global model per product — all stores together, not one per store.
    # This gives the model much more data to learn from and generalises better
    # to stores with limited individual history.
    global_model_p = fit_xgb_residual_model(
        train_pd_p, pretzel_features, "pretzel_target_resid",
        np.ones(len(train_pd_p), dtype=np.float32)
    )
    global_model_n = fit_xgb_residual_model(
        train_pd_n, nugget_features, "nugget_target_resid",
        np.ones(len(train_pd_n), dtype=np.float32)
    )

    # --- Calibration step ---
    # Run the model back over training data to measure systematic bias by time slot.
    # If it over-predicts at e.g. 10am on weekends, we learn a correction multiplier
    # and apply it when scoring the test weeks.
    train_pd_p["pred_pretzels_resid_raw"] = global_model_p.predict(
        train_pd_p[pretzel_features].to_numpy(dtype=np.float32)
    )
    # Convert log-residual back to real units:
    #   shrink the residual (don't fully trust the model's correction),
    #   add back to log(baseline), then expm1 to get back to raw sales scale
    train_pd_p["pred_pretzels_raw"] = np.expm1(
        PRETZEL_RESIDUAL_SHRINK * train_pd_p["pred_pretzels_resid_raw"] + np.log1p(train_pd_p["baseline_p"])
    )
    train_pd_p["pred_pretzels_raw"] = np.maximum(train_pd_p["pred_pretzels_raw"], 0.0)

    train_pd_n["pred_nuggets_resid_raw"] = global_model_n.predict(
        train_pd_n[nugget_features].to_numpy(dtype=np.float32)
    )
    train_pd_n["pred_nuggets_raw"] = np.expm1(
        NUGGET_RESIDUAL_SHRINK * train_pd_n["pred_nuggets_resid_raw"] + np.log1p(train_pd_n["baseline_n"])
    )
    train_pd_n["pred_nuggets_raw"] = np.maximum(train_pd_n["pred_nuggets_raw"], 0.0)

    calib_p = build_segment_calibration_pdf(
        train_pd_p, ACT_P, "pred_pretzels_raw", "pretzel_segment_mult",
        PRETZEL_CALIBRATION_MIN, PRETZEL_CALIBRATION_MAX
    )
    calib_n = build_segment_calibration_pdf(
        train_pd_n, ACT_N, "pred_nuggets_raw", "nugget_segment_mult",
        NUGGET_CALIBRATION_MIN, NUGGET_CALIBRATION_MAX
    )

    # --- Score the eval weeks ---
    # Same pipeline as training predictions, but now on the weeks we're evaluating against.

    test_pd_p["pred_pretzels_resid_raw"] = global_model_p.predict(
        test_pd_p[pretzel_features].to_numpy(dtype=np.float32)
    )
    test_pd_p["pred_pretzels_raw"] = np.expm1(
        PRETZEL_RESIDUAL_SHRINK * test_pd_p["pred_pretzels_resid_raw"] + np.log1p(test_pd_p["baseline_p"])
    )
    test_pd_p["pred_pretzels_raw"] = np.maximum(test_pd_p["pred_pretzels_raw"], 0.0)
    # Apply the calibration multiplier (e.g. scale down by 5% if we over-predict this slot type)
    test_pd_p = test_pd_p.merge(calib_p, on=["hour", "is_weekend"], how="left")
    test_pd_p["pretzel_segment_mult"] = test_pd_p["pretzel_segment_mult"].fillna(1.0)
    test_pd_p["pred_pretzels_raw"] = test_pd_p["pred_pretzels_raw"] * test_pd_p["pretzel_segment_mult"]

    # Zero suppression: snap to zero if the slot almost never sells AND
    # the baseline is near zero AND the raw prediction is near zero.
    # All three conditions must be true — any single strong signal overrides the suppression.
    test_pd_p["pred_pretzels"] = np.where(
        (
            (test_pd_p["pretzel_nonzero_rate_4w"] <= PRETZEL_ZERO_RATE_THRESHOLD) &
            (test_pd_p["baseline_p"] <= PRETZEL_BASELINE_ZERO_THRESHOLD) &
            (test_pd_p["pred_pretzels_raw"] <= PRETZEL_RAW_ZERO_CUTOFF)
        ),
        0.0,
        test_pd_p["pred_pretzels_raw"]
    )
    # Final cap: never predict above the 99.5th percentile seen in training
    test_pd_p["pred_pretzels"] = np.minimum(test_pd_p["pred_pretzels"], p995_pretzels)

    # Same pipeline for nuggets
    test_pd_n["pred_nuggets_resid_raw"] = global_model_n.predict(
        test_pd_n[nugget_features].to_numpy(dtype=np.float32)
    )
    test_pd_n["pred_nuggets_raw"] = np.expm1(
        NUGGET_RESIDUAL_SHRINK * test_pd_n["pred_nuggets_resid_raw"] + np.log1p(test_pd_n["baseline_n"])
    )
    test_pd_n["pred_nuggets_raw"] = np.maximum(test_pd_n["pred_nuggets_raw"], 0.0)
    test_pd_n = test_pd_n.merge(calib_n, on=["hour", "is_weekend"], how="left")
    test_pd_n["nugget_segment_mult"] = test_pd_n["nugget_segment_mult"].fillna(1.0)
    test_pd_n["pred_nuggets_raw"] = test_pd_n["pred_nuggets_raw"] * test_pd_n["nugget_segment_mult"]

    test_pd_n["pred_nuggets"] = np.where(
        (
            (test_pd_n["nugget_nonzero_rate_4w"] <= NUGGET_ZERO_RATE_THRESHOLD) &
            (test_pd_n["baseline_n"] <= NUGGET_BASELINE_ZERO_THRESHOLD) &
            (test_pd_n["pred_nuggets_raw"] <= NUGGET_RAW_ZERO_CUTOFF)
        ),
        0.0,
        test_pd_n["pred_nuggets_raw"]
    )
    test_pd_n["pred_nuggets"] = np.minimum(test_pd_n["pred_nuggets"], p995_nuggets)

    # Merge pretzel and nugget results into one row per store/hour slot
    pred_final_pd = (
        test_pd_p[[
            "Store id", "Time by hour", "forecast_week_start", "store_volume_segment",
            "hour", "day_of_week", "is_weekend", "is_holiday", "is_holiday_or_holiday_week",
            "is_opening_hour", "is_closing_hour", "baseline_p", ACT_P, "pred_pretzels"
        ]]
        .merge(
            test_pd_n[["Store id", "Time by hour", "baseline_n", ACT_N, "pred_nuggets"]],
            on=["Store id", "Time by hour"],
            how="inner"
        )
    )

    # Convert back to a Spark DataFrame for consistent handling downstream
    pred_final = spark.createDataFrame(pred_final_pd)

    # Store model metadata for the feature importance block later
    model_outputs = [{
        "eval_week_start": first_eval_week,
        "pretzel_features": pretzel_features,
        "nugget_features": nugget_features,
        "model_p": {"global": global_model_p},
        "model_n": {"global": global_model_n}
    }]

    # Free memory — pandas DataFrames across all stores for two weeks can be large
    del train_pd, test_pd, train_pd_p, train_pd_n, test_pd_p, test_pd_n, calib_p, calib_n
    gc.collect()

    return pred_final, pretzel_features, nugget_features, {"global": global_model_p}, {"global": global_model_n}, model_outputs


Block 4 — run weeks and materialise predictions

In [0]:
# COMMAND ----------

# =============================================================================
# RUN THE MODEL
# =============================================================================
# Train on all history before the two eval weeks, score both eval weeks.
# This is the single call that kicks off everything — it takes a while
# (mostly the XGBoost training step), so run it once and keep the outputs.

pred_final, pretzel_features, nugget_features, model_p, model_n, model_outputs = (
    run_single_train_two_week_score(full, eval_week_starts)
)

display(pred_final.limit(1000))


training cutoff: 2026-04-06 00:00:00
scoring window: 2026-04-06 00:00:00 to 2026-04-20 00:00:00
training-only excluded stores: 50
train rows after store exclusion: 11057475
test rows retained: 187642
train pretzels 99.5th percentile: 30.0
train nuggets 99.5th percentile: 66.0
store volume cutoffs: 8.481305361305362 14.974731934731935
pretzel features: 15
nugget features: 15


Store id,Time by hour,forecast_week_start,store_volume_segment,hour,day_of_week,is_weekend,is_holiday,is_holiday_or_holiday_week,is_opening_hour,is_closing_hour,baseline_p,Sales of Pretzels,pred_pretzels,baseline_n,Sales of Pretzel Nuggets,pred_nuggets
0d9e0782281d4ecf8caca19e4493955ef64499fd09784dfc59b568ec35b6ebed,2026-04-10T17:00:00.000Z,2026-04-06T00:00:00.000Z,low,17,6,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3b531ed420229f46c12681986fcb8bb44f2150fddd26d99b24cb695af3c2ec25,2026-04-06T10:00:00.000Z,2026-04-06T00:00:00.000Z,medium,10,2,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0
44c285bd79efd0a99fa7a7ea77b6830c3717b0cc50ed2169d2c6461d32cc6247,2026-04-07T17:00:00.000Z,2026-04-06T00:00:00.000Z,medium,17,3,0,0,0,0,0,5.5,2.0,5.110685233401176,10.5,10.0,9.833455939503382
a95e65fb34df20d1493a44cb5958aa66d0603f3ccb9f36d0c0123074af7748c6,2026-04-09T19:00:00.000Z,2026-04-06T00:00:00.000Z,high,19,5,0,0,0,0,0,3.5,3.0,2.8389148757819633,15.0,11.0,12.843069034883777
bf3711001c6aaff2f5aca734de539f3dd09776f8428d140fe7473aa34f7a2f82,2026-04-11T13:00:00.000Z,2026-04-06T00:00:00.000Z,high,13,7,1,0,0,0,0,19.5,6.0,19.06888746420732,58.75,39.0,55.66285225464601
5f98afe7fd9674cf5b22d4e762453a99d483b240667d86ceb236a3d9019a4569,2026-04-17T10:00:00.000Z,2026-04-13T00:00:00.000Z,high,10,6,0,0,0,1,0,0.5,0.0,0.5734679697908946,3.75,1.0,3.7942588658568974
ae5f9d3d0c628a39b7ab528806e86baba6523743be976dd3c4e9e22f0f23827b,2026-04-12T11:00:00.000Z,2026-04-06T00:00:00.000Z,low,11,1,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
e26207a4b918091fea2268e9913be5b2e24ce315685ab814b4863138ad893b99,2026-04-08T17:00:00.000Z,2026-04-06T00:00:00.000Z,medium,17,4,0,0,0,0,0,0.0,0.0,0.776030389137621,0.0,0.0,0.5787945483954409
8c56d91150da4025c6efc39848e3e4c4f1094c9da280dfddd65e1b90c5aa7763,2026-04-07T11:00:00.000Z,2026-04-06T00:00:00.000Z,low,11,3,0,0,0,0,0,0.0,0.0,0.0,0.5,0.0,0.39256883797533754
c7a1efdf21333843c08b805d461f51533071c8d4d927f8cc3955e392a94e95f6,2026-04-14T13:00:00.000Z,2026-04-13T00:00:00.000Z,low,13,3,0,0,0,0,0,2.5,4.0,2.0341092058349792,2.5,3.0,2.324297988170466


In [0]:
# COMMAND ----------

# =============================================================================
# ERROR COLUMNS + EVAL BASE
# =============================================================================

PRED_P = "pred_pretzels"
PRED_N = "pred_nuggets"

def add_error_cols(df):
    return (
        df
        .withColumn("actual_total", F.col(ACT_P) + F.col(ACT_N))
        .withColumn("pred_total", F.col(PRED_P) + F.col(PRED_N))
        .withColumn("err_p", F.col(PRED_P) - F.col(ACT_P))
        .withColumn("err_n", F.col(PRED_N) - F.col(ACT_N))
        .withColumn("err_total", F.col("pred_total") - F.col("actual_total"))
        .withColumn("ae_p", F.abs(F.col("err_p")))
        .withColumn("ae_n", F.abs(F.col("err_n")))
        .withColumn("ae_total", F.abs(F.col("err_total")))
        .withColumn("se_p", F.col("err_p") ** 2)
        .withColumn("se_n", F.col("err_n") ** 2)
        .withColumn("se_total", F.col("err_total") ** 2)
    )

def metrics_agg_exprs():
    return [
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.expr(f"try_divide(sum(ae_p), sum(`{ACT_P}`))").alias("wape_pretzels"),
        F.expr(f"try_divide(sum(ae_n), sum(`{ACT_N}`))").alias("wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("wape_total"),
    ]

def grouped_wape_prepared(df, group_cols):
    return (
        df
        .groupBy(*group_cols)
        .agg(
            F.count("*").alias("N"),
            F.mean("ae_p").alias("MAE_pretzels"),
            F.mean("ae_n").alias("MAE_nuggets"),
            F.mean("ae_total").alias("MAE_total"),
            F.expr(f"try_divide(sum(ae_p), sum(`{ACT_P}`))").alias("wape_pretzels"),
            F.expr(f"try_divide(sum(ae_n), sum(`{ACT_N}`))").alias("wape_nuggets"),
            F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("wape_total"),
            F.mean("err_p").alias("bias_pretzels"),
            F.mean("err_n").alias("bias_nuggets"),
        )
        .orderBy(*group_cols)
    )

eval_base = add_error_cols(pred_final)
_ = eval_base.count()


Block 5 — metrics helpers and eval base

In [0]:
# COMMAND ----------

# =============================================================================
# MAIN METRICS
# =============================================================================
# Top-level performance numbers across all stores, all hours, both eval weeks.
# These are the headline figures — start here when reviewing model performance.

# Overall metrics across the entire eval period
overall_metrics = eval_base.agg(*metrics_agg_exprs())
display(overall_metrics)

# COMMAND ----------

# Break down by week — useful for spotting if one week performed much worse than the other.
# A big week-on-week difference often points to a specific event or data issue.
weekly_metrics = (
    eval_base
    .groupBy("forecast_week_start")
    .agg(*metrics_agg_exprs())
    .orderBy("forecast_week_start")
)
display(weekly_metrics)

# COMMAND ----------

# WAPE summary — our primary metric. Lower is better.
# Shown overall and then split by week for context.
overall_wape_mae = (
    eval_base
    .agg(
        F.mean("ae_p").alias("mae_pretzels"),
        F.mean("ae_n").alias("mae_nuggets"),
        F.mean("ae_total").alias("mae_total"),
        F.expr("try_divide(sum(ae_p), sum(`Sales of Pretzels`))").alias("wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(`Sales of Pretzel Nuggets`))").alias("wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("wape_total")
    )
)
display(overall_wape_mae)

# COMMAND ----------

weekly_wape_mae = (
    eval_base
    .groupBy("forecast_week_start")
    .agg(
        F.mean("ae_p").alias("mae_pretzels"),
        F.mean("ae_n").alias("mae_nuggets"),
        F.mean("ae_total").alias("mae_total"),
        F.expr("try_divide(sum(ae_p), sum(`Sales of Pretzels`))").alias("wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(`Sales of Pretzel Nuggets`))").alias("wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("wape_total")
    )
    .orderBy("forecast_week_start")
)
display(weekly_wape_mae)

# COMMAND ----------

# Slice performance by different dimensions to understand where errors come from.
# By forecast week — are both weeks similar, or did something change?
# By hour — does the model struggle more at certain times of day?
# By day of week — are weekends harder to predict than weekdays?
# By weekend flag and holiday flag — do special days hurt accuracy?
# By hour × weekend — the most granular slice: which specific slot types are hardest?
display(grouped_wape_prepared(eval_base, ["forecast_week_start"]))
display(grouped_wape_prepared(eval_base, ["hour"]))
display(grouped_wape_prepared(eval_base, ["day_of_week"]))
display(grouped_wape_prepared(eval_base, ["is_weekend"]))
display(grouped_wape_prepared(eval_base, ["is_holiday"]))
display(grouped_wape_prepared(eval_base, ["hour", "is_weekend"]))

# COMMAND ----------

# Data sanity check — confirms the eval weeks have the expected row counts,
# volume, and time coverage before trusting the metrics above.
display(
    full
    .filter(F.col("forecast_week_start").isin(eval_week_starts))
    .groupBy("forecast_week_start")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("Sales of Pretzels")).alias("actual_p"),
        F.sum(F.col("Sales of Pretzel Nuggets")).alias("actual_n"),
        F.min("Time by hour").alias("min_ts"),
        F.max("Time by hour").alias("max_ts")
    )
    .orderBy("forecast_week_start")
)

# COMMAND ----------

# Day-by-day breakdown of the second eval week — useful for spotting if a specific
# day had data issues or unusually high/low sales that skewed overall metrics.
display(
    full
    .filter(F.col("forecast_week_start") == F.lit(eval_week_starts[1]))
    .withColumn("d", F.to_date("Time by hour"))
    .groupBy("d")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("Sales of Pretzels")).alias("actual_p"),
        F.sum(F.col("Sales of Pretzel Nuggets")).alias("actual_n")
    )
    .orderBy("d")
)


N,RMSE_pretzels,RMSE_nuggets,RMSE_total,MAE_pretzels,MAE_nuggets,MAE_total,Bias_pretzels,Bias_nuggets,Bias_total,wape_pretzels,wape_nuggets,wape_total
187642,3.560200441278803,5.620198235411867,7.309553410018806,1.465084387435875,3.1121206647342654,3.8731839531621706,-0.20687605633625245,-0.046465322102729065,-0.2533413784389816,0.4868824598984876,0.35362000846079183,0.32796184852490723


forecast_week_start,N,RMSE_pretzels,RMSE_nuggets,RMSE_total,MAE_pretzels,MAE_nuggets,MAE_total,Bias_pretzels,Bias_nuggets,Bias_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06T00:00:00.000Z,93821,4.329012411133449,6.010682938442294,8.115887738748034,1.529522749777456,3.2696769565737047,4.097245202991966,-0.4077888971736434,-0.83114966945303,-1.2389385666266715,0.47924041580600363,0.34249019942513637,0.3216469285503745
2026-04-13T00:00:00.000Z,93821,2.5709348316159515,5.200475653646702,6.402461114280548,1.4006460250942985,2.954564372894818,3.649122703332387,-0.005963215498862182,0.7382190252475703,0.732255809748708,0.4955109924938902,0.36681149980993133,0.33535442755026473


mae_pretzels,mae_nuggets,mae_total,wape_pretzels,wape_nuggets,wape_total
1.465084387435875,3.1121206647342654,3.8731839531621706,0.4868824598984876,0.35362000846079183,0.32796184852490723


forecast_week_start,mae_pretzels,mae_nuggets,mae_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06T00:00:00.000Z,1.529522749777456,3.2696769565737047,4.097245202991966,0.47924041580600363,0.34249019942513637,0.3216469285503745
2026-04-13T00:00:00.000Z,1.4006460250942985,2.954564372894818,3.649122703332387,0.4955109924938902,0.36681149980993133,0.33535442755026473


forecast_week_start,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
2026-04-06T00:00:00.000Z,93821,1.529522749777456,3.2696769565737047,4.097245202991966,0.47924041580600363,0.34249019942513637,0.3216469285503745,-0.4077888971736434,-0.83114966945303
2026-04-13T00:00:00.000Z,93821,1.4006460250942985,2.954564372894818,3.649122703332387,0.4955109924938902,0.36681149980993133,0.33535442755026473,-0.005963215498862182,0.7382190252475703


...

is_weekend,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,134030,1.3542547023006346,2.9170681932313247,3.6273733620159923,0.5419525789721548,0.42451931735777076,0.3871136853918566,-0.09460238379749478,0.301844432706863
1,53612,1.7421586002739815,3.599751843491612,4.487710431027635,0.4065916474163258,0.264222045532459,0.2505875612194399,-0.4875602376831472,-0.9172397091267083


is_holiday,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,187642,1.465084387435875,3.1121206647342654,3.8731839531621706,0.4868824598984876,0.35362000846079183,0.32796184852490723,-0.20687605633625283,-0.046465322102729176


hour,is_weekend,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
10,0,10310,0.6671477311982075,1.0813341322708045,1.4259656657930273,0.8114065245550925,0.6568405646445528,0.5776701773802009,-0.1264447548801238,-0.094553515721329
10,1,4124,0.6465528647601242,1.1576397028273147,1.4915873995282125,0.6611415854874169,0.49217589015049956,0.4479215346722747,-0.1496284390014527,-0.1780523775988304
11,0,10310,1.5661983764484446,2.6744457147446656,3.551208069025753,0.624903454380165,0.514518022037609,0.4609403783366131,-0.27988423601473594,0.08403750575473154
11,1,4124,1.7153211270547457,3.161891051418811,4.066715448881066,0.4828328665602192,0.33894722507996095,0.3157097720565023,-0.43202263693393517,-0.6326758347292074
12,0,10310,1.7947225672715619,3.7391261324648064,4.72475751693131,0.5384428828334004,0.4748930168115618,0.42159777396584625,-0.09753994352066962,0.39449962120223014
12,1,4124,2.465248322293781,4.889596194365764,6.200142689596233,0.407564004054502,0.27987084948736174,0.2636155312324848,-0.7541321761143719,-1.3489102564121183
13,0,10310,1.9488875009447437,4.327371704552788,5.417536878792116,0.5388315938519793,0.481452089976467,0.42979120346840294,-0.07620578878289694,0.637467502286503
13,1,4124,2.738376194593982,5.678103452317919,7.062847642360632,0.37726543149948494,0.25063952216553137,0.23611338816234667,-0.7907609959223145,-1.7547634184482748
14,0,10310,2.015991250309602,4.694190489715745,5.868580795225833,0.5460075601095962,0.4873581788325798,0.4404468741721628,-0.036270869787573934,0.7733866669141284
14,1,4124,2.9984056268205626,6.227358344782253,7.815248846764405,0.3718924753385865,0.24477340653719035,0.23326399539738296,-1.0928897167868399,-1.9518490341753236


forecast_week_start,rows,actual_p,actual_n,min_ts,max_ts
2026-04-06T00:00:00.000Z,98371,305810.0,914548.0,2026-04-06T10:00:00.000Z,2026-04-12T22:00:00.000Z
2026-04-13T00:00:00.000Z,98371,270812.0,771610.0,2026-04-13T10:00:00.000Z,2026-04-19T22:00:00.000Z


d,rows,actual_p,actual_n
2026-04-13,14053,27472.0,69746.0
2026-04-14,14053,27807.0,67002.0
2026-04-15,14053,27832.0,71586.0
2026-04-16,14053,30132.0,78244.0
2026-04-17,14053,39433.0,111175.0
2026-04-18,14053,66238.0,215686.0
2026-04-19,14053,51898.0,158171.0


Block 6 — main metrics

Block 7 — store performance

In [0]:
# COMMAND ----------

# =============================================================================
# STORE-LEVEL PERFORMANCE
# =============================================================================
# Break down WAPE and MAE by individual store. This shows which stores the model
# handles well and which it struggles with. The worst performers are worth
# investigating — they might have unusual patterns, intermittent trading, or
# data quality issues. The best performers are useful for understanding what
# conditions the model thrives in.

store_perf = grouped_wape_prepared(eval_base, ["Store id"])
_ = store_perf.count()

# Show the worst-performing stores first (highest WAPE = biggest relative errors)
display(store_perf.orderBy(F.desc("wape_total")).limit(200))
# Then the best (lowest WAPE = most accurate predictions)
display(store_perf.orderBy(F.asc("wape_total")).limit(200))


Store id,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
10202675f3ef7918a03b1e0f46648e41dea9c898e5b05ec5f08fdbe670f428f4,182,1.77441748846939,2.7522898866266585,4.522816899718112,64.58879658028579,null,164.63053514973927,1.7705270130914559,2.7522898866266585
4f6fa8202180a364127be274f7bd5f3422f6f2cbe2c50c98d7b8c5a20da082fe,182,0.006618875796346275,0.1855115030607669,0.18114136786810217,1.2046353949350221,null,32.967728951994594,-0.004370135192664714,0.1855115030607669
034226c880b7d19639ea8db9a2550d175ca111d3786a2db5703a5ed807523186,182,0.1579963156325623,0.335463958401902,0.4722020536065969,3.1950366050140375,2.105325532039523,2.261599309378964,0.09206224969849634,0.23880412586868902
77ece305ff3afe25e0f0b989b1454903c7a0949cc844f0a4cf60f42597b0fed2,182,1.0610499205447228,2.8188279914241776,3.6321574902541824,4.827777138478489,1.8192435973021288,2.0529585814480162,0.9160309188354934,1.205152703644981
e944c09a9b8041ad593a68f48001005418b83ae2ba6f0f48543ffa2e482dd993,182,1.124378907690917,2.6534531687539253,3.7269923127633575,1.6912145553698092,2.063796909030831,1.910739720909665,0.2717037429143172,1.3233454475943456
01e3aef8262cf46d2f98a1942f5322e7500a6b487c8c00832742c9bfc17190c9,182,2.3643137164417656,3.0390364120636013,4.777566278049184,4.13754900377309,1.4109811913152435,1.7530585939615957,2.0093123103337507,-0.46383120165803493
1224de786adda26ed0b074454e45cb606097a51efa9d6416ebd25881d400c9cc,182,4.929245066337989,15.584196828929338,20.25138501176539,1.7186256744703334,1.7649805991693461,1.7312128098362145,2.872170947452841,10.74988119542633
bb441d21606a002a5307232e9d1c2002a8f6f782bfd8cb1cc23776665835f13f,182,0.028787460398466212,0.0,0.028787460398466212,1.3098294481302126,null,1.3098294481302126,-0.015168583557577742,0.0
f9ba7d24ac2b38dbb189e9c3cc0bf3ab1fc9abbb363d351eaa871a2f43f99846,182,1.2484138191948966,0.07998845936302401,1.3284022785579208,1.1248084905617384,null,1.196877300482879,-0.9713664005853232,0.07998845936302401
25a811f8ca6bf82221778dfe5c50ff29730a5236a7feb8beb61da4c6ccdb1fff,182,0.8819579445862186,4.701322540760915,5.2037161957134686,2.1988540536259147,1.1516025604555675,1.1606327789458961,0.4960949083988772,3.175327608201886


Store id,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
962bee07fdc59c5b9e09856cd5424406efad5dc07e3279b7311ee7a7757083dc,182,0.0,0.0,0.0,null,null,null,0.0,0.0
2f5cb1039e68a6721d0c7ff0393e1090d4ffffdb8fe4b5966a8deb88f9f8366f,182,0.0,0.0,0.0,null,null,null,0.0,0.0
0a8d624efa96f181bde35fc14a4169b92cbedc36a662f254cc5aa14b0cc9612e,182,0.26505267840937147,0.33853633590281235,0.6035890143121839,null,null,null,0.26505267840937147,0.33853633590281235
2aa52279231e240a44ef804bd87935203be0cfe17d07451f0dca106131f24490,182,0.0,0.0,0.0,null,null,null,0.0,0.0
bd9f8f5a398b5f1aae6b11520b1a554f951863fcdd0db46a1a4d43fa04b415fd,182,0.0,0.0,0.0,null,null,null,0.0,0.0
b09faa149269da96b83507f787f808393442b4de3f954e5940ced28409eb1019,182,0.0,0.0,0.0,null,null,null,0.0,0.0
56385bb0e9ee856185c965e5bbc925aa4e021e22063c45a5909840b56c4d4932,182,0.0,0.0,0.0,null,null,null,0.0,0.0
1b4bd31c89d6db94ad62afc33ba525cd093a3607a8efa485ff9be1e7c8efb1fc,182,0.00896363365121499,0.0,0.00896363365121499,null,null,null,0.00896363365121499,0.0
6d5ba39726ec51d7772f8068a0912e27f5f07b014685ce5def922b0b0f4c68fb,182,0.0,0.0,0.0,null,null,null,0.0,0.0
c076686b0e5b45747db32bc7e3fd3887c80b7a1bc084fbbe4d04c3ba29580530,182,0.0,0.0,0.0,null,null,null,0.0,0.0


Block 8 — bucket diagnostics

In [0]:
# COMMAND ----------

# =============================================================================
# BUCKET DIAGNOSTICS — ERROR BY ACTUAL SALES VOLUME
# =============================================================================
# Slice errors by how much actually sold. This tells us whether the model
# performs differently depending on demand level.
# Common findings:
#   - Rows with actual = 0 often have non-zero predictions (false alarms)
#   - High-volume rows (21+) tend to have higher absolute error but lower % error
#   - The middle buckets (3-10) are often where the model adds most value

diagnostic_df = (
    eval_base
    .withColumn(
        "pretzel_bucket",
        F.when(F.col(ACT_P) == 0, "0")
         .when(F.col(ACT_P) <= 2, "1-2")
         .when(F.col(ACT_P) <= 5, "3-5")
         .when(F.col(ACT_P) <= 10, "6-10")
         .when(F.col(ACT_P) <= 20, "11-20")
         .otherwise("21+")
    )
    .withColumn(
        "nugget_bucket",
        F.when(F.col(ACT_N) == 0, "0")
         .when(F.col(ACT_N) <= 2, "1-2")
         .when(F.col(ACT_N) <= 5, "3-5")
         .when(F.col(ACT_N) <= 10, "6-10")
         .when(F.col(ACT_N) <= 20, "11-20")
         .otherwise("21+")
    )
)

display(grouped_wape_prepared(diagnostic_df, ["pretzel_bucket"]))
display(grouped_wape_prepared(diagnostic_df, ["nugget_bucket"]))


pretzel_bucket,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,85614,0.3891022457308567,0.9636758639662271,1.1826633179169421,null,0.5838314787361891,0.7165024045581934,0.3891022457308567,0.3037525572945551
1-2,33283,1.3045958658510453,3.719604572658478,4.335292787952592,0.8818926030368093,0.41817831407992745,0.4178960031436022,0.8612162826208979,0.592614875716159
11-20,10737,4.778077305462089,7.2510192977450245,10.087617547154293,0.3414525150501937,0.2642825182370177,0.24348577132996949,-3.5025820866628643,-2.644004704571555
21+,2356,10.183250854890119,9.257605289744047,16.75073182195238,0.3622980476604268,0.24666845426067013,0.25519890439605936,-9.680616404259178,-4.875132880559929
3-5,32736,1.7447688092139544,4.528404079069468,5.274354641164473,0.45193381814348454,0.36178340593581526,0.322047528114434,-0.16797684707335184,0.16368046688177912
6-10,22916,2.8699246804339933,5.662151037167322,7.016545159199402,0.3811508265873775,0.30800785525538693,0.27077559057456,-1.5221440035256126,-0.8697883794263916


nugget_bucket,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,68349,0.1631036033310687,0.36807077969240953,0.496032331043425,1.904333478659927,null,5.7914782703257695,0.061375401348905366,0.36807077969240953
1-2,11571,1.1810690361535379,2.526427176904877,3.0334291485377465,0.8013457146319096,1.6536536295942035,1.010589907800595,0.07503849029578982,2.1338457927695873
11-20,32090,2.324713511566571,4.647816432405042,5.810791708506628,0.4619341563898032,0.31603327827511146,0.2943768505100416,-0.3445873954933224,-0.2810474045765263
21+,24700,3.4458666348430276,8.733080032683313,10.628280067120942,0.37255058163627236,0.25664534586171805,0.24558609328422012,-1.3017708291741412,-5.33163540890077
3-5,20605,1.5813981610346262,2.9044502087890014,3.651522159682488,0.6631264827245404,0.723269319250911,0.5705070752662051,0.031198582744836736,2.0329922256373174
6-10,30327,1.9058784268448115,3.458050071278087,4.403416862298166,0.5631565747641896,0.4396386569729882,0.39141683988931397,-0.04329978027933426,1.3273174281702498


Block 9 — spike diagnostics

In [0]:
# COMMAND ----------

# =============================================================================
# SPIKE DIAGNOSTICS
# =============================================================================
# Flag rows where actual sales were at least 1.5× what the model predicted —
# these are the "surprise spike" cases where demand came in much higher than expected.
# Looking at WAPE for spiked vs. non-spiked rows shows how much of the total error
# is driven by these hard-to-predict demand spikes.
# If spike rows dominate error, it suggests the model needs better event/holiday signals.

spike_df = (
    eval_base
    .withColumn(
        "pretzel_spike_flag",
        F.when(F.col(ACT_P) >= 1.5 * F.col(PRED_P), 1).otherwise(0)
    )
    .withColumn(
        "nugget_spike_flag",
        F.when(F.col(ACT_N) >= 1.5 * F.col(PRED_N), 1).otherwise(0)
    )
)

display(grouped_wape_prepared(spike_df, ["pretzel_spike_flag"]))
display(grouped_wape_prepared(spike_df, ["nugget_spike_flag"]))


pretzel_spike_flag,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,90777,1.6491329620895614,4.476626728785019,5.2291143670361055,0.44606873208346726,0.34665567765750416,0.3148022020988678,0.9516660031130317,0.7981261522854474
1,96865,1.2926033317259975,1.8333742963211561,2.6024742522357895,0.5466883016532277,0.3706598399833627,0.3559830267723975,-1.2926033317259975,-0.8379739193311964


nugget_spike_flag,N,MAE_pretzels,MAE_nuggets,MAE_total,wape_pretzels,wape_nuggets,wape_total,bias_pretzels,bias_nuggets
0,105446,2.067272131958381,3.7412320376413146,4.741840735333715,0.47456774952155495,0.3093793429526048,0.28827851989723546,-0.0345819752754756,1.7141261533006475
1,82196,0.6925615285507747,2.3050585470210283,2.758818496779132,0.5406006343791557,0.5035482955564072,0.4708903596640478,-0.4279053238375259,-2.3050585470210283


Block 10 — feature lists

In [0]:
# COMMAND ----------

import pandas as pd

# =============================================================================
# FEATURE LISTS
# =============================================================================
# Print and display the exact set of features that went into training.
# Useful for documentation and for spotting if an expected feature was silently
# dropped (e.g. because it wasn't present in the data for a given eval week).

pretzel_feature_list = pretzel_features
nugget_feature_list = nugget_features

print("Number of pretzel features:", len(pretzel_feature_list))
print("\nPretzel Features:\n")
for f in pretzel_feature_list:
    print(f)

display(pd.DataFrame({"Pretzel_Model_Features": pretzel_feature_list}))

print("\n\nNumber of nugget features:", len(nugget_feature_list))
print("\nNugget Features:\n")
for f in nugget_feature_list:
    print(f)

display(pd.DataFrame({"Nugget_Model_Features": nugget_feature_list}))


Number of pretzel features: 15

Pretzel Features:

baseline_p
pretzel_nonzero_rate_4w
avg_sales_store_hour_dow_pretzels
Lag_1_week_Pretzels
SameSlot_4week_std_Pretzels
pretzel_baseline_minus_lag1
pretzel_recent_volatility
day_of_week
is_weekend
is_holiday
is_holiday_or_holiday_week
hour_sin
hour_cos
is_opening_hour
is_closing_hour


Pretzel_Model_Features
baseline_p
pretzel_nonzero_rate_4w
avg_sales_store_hour_dow_pretzels
Lag_1_week_Pretzels
SameSlot_4week_std_Pretzels
pretzel_baseline_minus_lag1
pretzel_recent_volatility
day_of_week
is_weekend
is_holiday




Number of nugget features: 15

Nugget Features:

baseline_n
nugget_nonzero_rate_4w
avg_sales_store_hour_dow_nuggets
Lag_1_week_Nuggets
SameSlot_4week_std_Nuggets
nugget_baseline_minus_lag1
nugget_recent_volatility
day_of_week
is_weekend
is_holiday
is_holiday_or_holiday_week
hour_sin
hour_cos
is_opening_hour
is_closing_hour


Nugget_Model_Features
baseline_n
nugget_nonzero_rate_4w
avg_sales_store_hour_dow_nuggets
Lag_1_week_Nuggets
SameSlot_4week_std_Nuggets
nugget_baseline_minus_lag1
nugget_recent_volatility
day_of_week
is_weekend
is_holiday


Block 11 — feature importance

In [0]:
# COMMAND ----------

import pandas as pd

# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================
# XGBoost tracks how much each feature contributed to reducing prediction error
# across all the trees it built. Higher importance = the model relied on that
# feature more heavily.
# This is useful for understanding what's driving predictions and for deciding
# which features to add or remove in future iterations.
# Note: importance scores are relative, not absolute — compare features against
# each other, not against some external benchmark.

pretzel_importance_dfs = []
nugget_importance_dfs = []

for out in model_outputs:
    ws = out["eval_week_start"]

    for seg_name, seg_model in out["model_p"].items():
        pretzel_importance_dfs.append(
            pd.DataFrame({
                "forecast_week_start": ws,
                "segment": seg_name,
                "feature": out["pretzel_features"],
                "feature_importance": list(seg_model.feature_importances_)
            })
        )

    for seg_name, seg_model in out["model_n"].items():
        nugget_importance_dfs.append(
            pd.DataFrame({
                "forecast_week_start": ws,
                "segment": seg_name,
                "feature": out["nugget_features"],
                "feature_importance": list(seg_model.feature_importances_)
            })
        )

pretzel_importance_all = pd.concat(pretzel_importance_dfs, ignore_index=True)
nugget_importance_all = pd.concat(nugget_importance_dfs, ignore_index=True)

pretzel_importance_all["abs_feature_importance"] = pretzel_importance_all["feature_importance"].abs()
nugget_importance_all["abs_feature_importance"] = nugget_importance_all["feature_importance"].abs()

# Raw importance per week and segment — highest importance features at the top
display(
    pretzel_importance_all.sort_values(
        ["forecast_week_start", "segment", "abs_feature_importance"],
        ascending=[True, True, False]
    ).drop(columns="abs_feature_importance")
)
display(
    nugget_importance_all.sort_values(
        ["forecast_week_start", "segment", "abs_feature_importance"],
        ascending=[True, True, False]
    ).drop(columns="abs_feature_importance")
)

# COMMAND ----------

# Average importance across weeks — smooths out week-to-week variation
# to show which features are consistently important vs. occasionally important
pretzel_importance_avg = (
    pretzel_importance_all
    .groupby(["segment", "feature"], as_index=False)
    .agg(
        mean_feature_importance=("feature_importance", "mean"),
        mean_abs_feature_importance=("abs_feature_importance", "mean")
    )
    .sort_values(["segment", "mean_abs_feature_importance"], ascending=[True, False])
)

nugget_importance_avg = (
    nugget_importance_all
    .groupby(["segment", "feature"], as_index=False)
    .agg(
        mean_feature_importance=("feature_importance", "mean"),
        mean_abs_feature_importance=("abs_feature_importance", "mean")
    )
    .sort_values(["segment", "mean_abs_feature_importance"], ascending=[True, False])
)

display(pretzel_importance_avg)
display(nugget_importance_avg)

# COMMAND ----------

# Stability of feature importance across weeks.
# A feature with high mean importance but also high std is inconsistent —
# it matters a lot some weeks and barely at all in others.
# Features with low std are reliable contributors.
pretzel_importance_stability = (
    pretzel_importance_all
    .groupby(["segment", "feature"], as_index=False)
    .agg(
        mean_feature_importance=("feature_importance", "mean"),
        std_feature_importance=("feature_importance", "std"),
        mean_abs_feature_importance=("abs_feature_importance", "mean")
    )
    .sort_values(["segment", "mean_abs_feature_importance"], ascending=[True, False])
)

nugget_importance_stability = (
    nugget_importance_all
    .groupby(["segment", "feature"], as_index=False)
    .agg(
        mean_feature_importance=("feature_importance", "mean"),
        std_feature_importance=("feature_importance", "std"),
        mean_abs_feature_importance=("abs_feature_importance", "mean")
    )
    .sort_values(["segment", "mean_abs_feature_importance"], ascending=[True, False])
)

display(pretzel_importance_stability)
display(nugget_importance_stability)


forecast_week_start,segment,feature,feature_importance
2026-04-06T00:00:00.000Z,global,baseline_p,0.21241808
2026-04-06T00:00:00.000Z,global,avg_sales_store_hour_dow_pretzels,0.17069165
2026-04-06T00:00:00.000Z,global,pretzel_recent_volatility,0.15953249
2026-04-06T00:00:00.000Z,global,SameSlot_4week_std_Pretzels,0.12766886
2026-04-06T00:00:00.000Z,global,pretzel_baseline_minus_lag1,0.056674615
2026-04-06T00:00:00.000Z,global,day_of_week,0.04885016
2026-04-06T00:00:00.000Z,global,pretzel_nonzero_rate_4w,0.041371927
2026-04-06T00:00:00.000Z,global,Lag_1_week_Pretzels,0.038213328
2026-04-06T00:00:00.000Z,global,is_holiday,0.028769106
2026-04-06T00:00:00.000Z,global,is_closing_hour,0.02413129


forecast_week_start,segment,feature,feature_importance
2026-04-06T00:00:00.000Z,global,nugget_baseline_minus_lag1,0.14686711
2026-04-06T00:00:00.000Z,global,avg_sales_store_hour_dow_nuggets,0.14180331
2026-04-06T00:00:00.000Z,global,nugget_recent_volatility,0.13108566
2026-04-06T00:00:00.000Z,global,baseline_n,0.085011445
2026-04-06T00:00:00.000Z,global,SameSlot_4week_std_Nuggets,0.0793439
2026-04-06T00:00:00.000Z,global,day_of_week,0.07254786
2026-04-06T00:00:00.000Z,global,nugget_nonzero_rate_4w,0.061395135
2026-04-06T00:00:00.000Z,global,is_holiday,0.0591706
2026-04-06T00:00:00.000Z,global,is_weekend,0.04352294
2026-04-06T00:00:00.000Z,global,hour_cos,0.037602916


segment,feature,mean_feature_importance,mean_abs_feature_importance
global,baseline_p,0.21241808,0.21241808
global,avg_sales_store_hour_dow_pretzels,0.17069165,0.17069165
global,pretzel_recent_volatility,0.15953249,0.15953249
global,SameSlot_4week_std_Pretzels,0.12766886,0.12766886
global,pretzel_baseline_minus_lag1,0.056674615,0.056674615
global,day_of_week,0.04885016,0.04885016
global,pretzel_nonzero_rate_4w,0.041371927,0.041371927
global,Lag_1_week_Pretzels,0.038213328,0.038213328
global,is_holiday,0.028769106,0.028769106
global,is_closing_hour,0.02413129,0.02413129


segment,feature,mean_feature_importance,mean_abs_feature_importance
global,nugget_baseline_minus_lag1,0.14686711,0.14686711
global,avg_sales_store_hour_dow_nuggets,0.14180331,0.14180331
global,nugget_recent_volatility,0.13108566,0.13108566
global,baseline_n,0.085011445,0.085011445
global,SameSlot_4week_std_Nuggets,0.0793439,0.0793439
global,day_of_week,0.07254786,0.07254786
global,nugget_nonzero_rate_4w,0.061395135,0.061395135
global,is_holiday,0.0591706,0.0591706
global,is_weekend,0.04352294,0.04352294
global,hour_cos,0.037602916,0.037602916


segment,feature,mean_feature_importance,std_feature_importance,mean_abs_feature_importance
global,baseline_p,0.21241808,null,0.21241808
global,avg_sales_store_hour_dow_pretzels,0.17069165,null,0.17069165
global,pretzel_recent_volatility,0.15953249,null,0.15953249
global,SameSlot_4week_std_Pretzels,0.12766886,null,0.12766886
global,pretzel_baseline_minus_lag1,0.056674615,null,0.056674615
global,day_of_week,0.04885016,null,0.04885016
global,pretzel_nonzero_rate_4w,0.041371927,null,0.041371927
global,Lag_1_week_Pretzels,0.038213328,null,0.038213328
global,is_holiday,0.028769106,null,0.028769106
global,is_closing_hour,0.02413129,null,0.02413129


segment,feature,mean_feature_importance,std_feature_importance,mean_abs_feature_importance
global,nugget_baseline_minus_lag1,0.14686711,null,0.14686711
global,avg_sales_store_hour_dow_nuggets,0.14180331,null,0.14180331
global,nugget_recent_volatility,0.13108566,null,0.13108566
global,baseline_n,0.085011445,null,0.085011445
global,SameSlot_4week_std_Nuggets,0.0793439,null,0.0793439
global,day_of_week,0.07254786,null,0.07254786
global,nugget_nonzero_rate_4w,0.061395135,null,0.061395135
global,is_holiday,0.0591706,null,0.0591706
global,is_weekend,0.04352294,null,0.04352294
global,hour_cos,0.037602916,null,0.037602916


Block 12 — weekly summary

In [0]:
# COMMAND ----------

# =============================================================================
# WEEKLY SUMMARY
# =============================================================================
# A comprehensive metric table broken down by forecast week.
# Includes RMSE, MAE, MAPE, Bias, WAPE, and raw volume — everything needed
# to present a full picture of performance for each week in the eval window.
# WAPE is the headline; Bias tells you which direction errors lean.

weekly_summary = (
    eval_base
    .groupBy("forecast_week_start")
    .agg(
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.mean(F.when(F.col(ACT_P) != 0, F.col("ae_p") / F.col(ACT_P))).alias("MAPE_pretzels"),
        F.mean(F.when(F.col(ACT_N) != 0, F.col("ae_n") / F.col(ACT_N))).alias("MAPE_nuggets"),
        F.mean(F.when(F.col("actual_total") != 0, F.col("ae_total") / F.col("actual_total"))).alias("MAPE_total"),
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume")
    )
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .orderBy("forecast_week_start")
)

display(weekly_summary)


forecast_week_start,N,RMSE_pretzels,RMSE_nuggets,RMSE_total,MAE_pretzels,MAE_nuggets,MAE_total,Bias_pretzels,Bias_nuggets,Bias_total,MAPE_pretzels,MAPE_nuggets,MAPE_total,sum_ae_p,sum_ae_n,sum_ae_total,pretzel_volume,nugget_volume,total_volume,wape_pretzels,wape_nuggets,wape_total
2026-04-06T00:00:00.000Z,93821,4.329012411133449,6.01068293844229,8.11588773874803,1.5295227497774555,3.2696769565737087,4.097245202991962,-0.40778889717364264,-0.8311496694530287,-1.238938566626671,0.5693141286108926,0.48599978066873306,0.44780620028007767,143501.35390687065,306764.3617427019,384407.64218990883,299435.0,895688.0,1195123.0,0.47924041580600346,0.34249019942513675,0.3216469285503742
2026-04-13T00:00:00.000Z,93821,2.5709348316159493,5.2004756536467,6.402461114280547,1.400646025094299,2.9545643728948154,3.649122703332383,-0.005963215498862183,0.7382190252475702,0.7322558097487082,0.6334141449212358,0.6360938438696652,0.5672240743153284,131410.01072037223,277200.1840293645,342364.34114934754,265201.0,755702.0,1020903.0,0.4955109924938904,0.366811499809931,0.33535442755026434


Block 13 — baseline function

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark import StorageLevel
from datetime import timedelta

ACT_P = "Sales of Pretzels"
ACT_N = "Sales of Pretzel Nuggets"

# =============================================================================
# BASELINE COMPARISON FUNCTION
# =============================================================================
# To know whether the XGBoost model is actually adding value, we need to compare
# it against simple baselines — naive rules a non-ML system might use.
# This function computes several such baselines for a given eval week:
#   - Last week's value for the same store/hour
#   - Same slot from exactly one year ago (364 days, to land on the same weekday)
#   - Rolling averages over the last 1, 2, 3, 4 weeks
#   - Training-set store/hour average (how this slot has behaved historically)
#   - Training-set store/hour/day-of-week average (more specific historical average)
# If the model can't beat these, it's not worth the complexity.

def run_one_week_baselines(full_df, eval_week_start):
    eval_week_end = eval_week_start + timedelta(days=7)

    eligible_df = full_df.filter(~F.col("Store id").isin(TRAIN_EXCLUDED_STORES))

    train_df = eligible_df.filter(F.col("forecast_week_start") < F.lit(eval_week_start))
    test_df = eligible_df.filter(
        (F.col("forecast_week_start") == F.lit(eval_week_start)) &
        (F.col("Time by hour") >= F.lit(eval_week_start)) &
        (F.col("Time by hour") < F.lit(eval_week_end))
    )

    # Build a historical lookup table for the year-ago baseline
    hist = eligible_df.select(
        "Store id",
        F.col("Time by hour").alias("hist_ts"),
        F.col(ACT_P).alias("hist_p"),
        F.col(ACT_N).alias("hist_n")
    )

    print(f"\nbaselines for week starting {eval_week_start}")

    # Training-set averages — computed from all training data, not just recent lags
    train_store_hour_avg = train_df.groupBy("Store id", "hour").agg(
        F.mean(ACT_P).alias("bl_train_store_hour_avg_p"),
        F.mean(ACT_N).alias("bl_train_store_hour_avg_n")
    )

    train_store_hour_dow_avg = train_df.groupBy("Store id", "hour", "day_of_week").agg(
        F.mean(ACT_P).alias("bl_train_store_hour_dow_avg_p"),
        F.mean(ACT_N).alias("bl_train_store_hour_dow_avg_n")
    )

    def avg_available(cols):
        # Average over whichever lag weeks are non-null — handles stores with limited history
        return (
            sum(F.coalesce(F.col(c), F.lit(0.0)) for c in cols) /
            F.greatest(
                sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in cols),
                F.lit(1)
            )
        )

    # Join year-ago data and compute rolling lag averages
    baseline_week = (
        test_df.alias("t")
        .join(
            hist.alias("hy"),
            (
                (F.col("t.Store id") == F.col("hy.Store id")) &
                (F.col("hy.hist_ts") == F.col("t.Time by hour") - F.expr("INTERVAL 364 DAYS"))
            ),
            "left"
        )
        .select(
            "t.*",
            F.col("hy.hist_p").alias("bl_lastyear_p"),
            F.col("hy.hist_n").alias("bl_lastyear_n")
        )
        .join(train_store_hour_avg, ["Store id", "hour"], "left")
        .join(train_store_hour_dow_avg, ["Store id", "hour", "day_of_week"], "left")
        .withColumn("bl_lastweek_p", F.col("Lag_1_week_Pretzels"))
        .withColumn("bl_lastweek_n", F.col("Lag_1_week_Nuggets"))
        .withColumn("bl_avg_1w_p", F.col("Lag_1_week_Pretzels"))
        .withColumn("bl_avg_1w_n", F.col("Lag_1_week_Nuggets"))
        .withColumn("bl_avg_2w_p", avg_available(["Lag_1_week_Pretzels", "Lag_2_week_Pretzels"]))
        .withColumn("bl_avg_2w_n", avg_available(["Lag_1_week_Nuggets", "Lag_2_week_Nuggets"]))
        .withColumn(
            "bl_avg_3w_p",
            avg_available(["Lag_1_week_Pretzels", "Lag_2_week_Pretzels", "Lag_3_week_Pretzels"])
        )
        .withColumn(
            "bl_avg_3w_n",
            avg_available(["Lag_1_week_Nuggets", "Lag_2_week_Nuggets", "Lag_3_week_Nuggets"])
        )
    )

    # Add the 4-week lag and average (requires a separate join for lag 4)
    baseline_week = (
        baseline_week.alias("t")
        .join(
            hist.alias("h4"),
            (
                (F.col("t.Store id") == F.col("h4.Store id")) &
                (F.col("h4.hist_ts") == F.col("t.Time by hour") - F.expr("INTERVAL 28 DAYS"))
            ),
            "left"
        )
        .select(
            "t.*",
            F.col("h4.hist_p").alias("bl_lag4_p_tmp"),
            F.col("h4.hist_n").alias("bl_lag4_n_tmp")
        )
        .withColumn("Lag_4_week_Pretzels", F.col("bl_lag4_p_tmp"))
        .withColumn("Lag_4_week_Nuggets", F.col("bl_lag4_n_tmp"))
        .drop("bl_lag4_p_tmp", "bl_lag4_n_tmp")
        .withColumn(
            "bl_avg_4w_p",
            avg_available([
                "Lag_1_week_Pretzels", "Lag_2_week_Pretzels",
                "Lag_3_week_Pretzels", "Lag_4_week_Pretzels"
            ])
        )
        .withColumn(
            "bl_avg_4w_n",
            avg_available([
                "Lag_1_week_Nuggets", "Lag_2_week_Nuggets",
                "Lag_3_week_Nuggets", "Lag_4_week_Nuggets"
            ])
        )
        .select(
            "Store id", "Time by hour", "forecast_week_start", "hour", "day_of_week",
            "is_weekend", "is_holiday", ACT_P, ACT_N,
            "bl_lastweek_p", "bl_lastweek_n",
            "bl_lastyear_p", "bl_lastyear_n",
            "bl_avg_1w_p", "bl_avg_1w_n",
            "bl_avg_2w_p", "bl_avg_2w_n",
            "bl_avg_3w_p", "bl_avg_3w_n",
            "bl_avg_4w_p", "bl_avg_4w_n",
            "bl_train_store_hour_avg_p", "bl_train_store_hour_avg_n",
            "bl_train_store_hour_dow_avg_p", "bl_train_store_hour_dow_avg_n"
        )
    )

    _ = baseline_week.count()
    compare_total_vs_distinct(baseline_week, ["Store id", "Time by hour"], "baseline_week")

    return baseline_week


Block 14 — baseline runs

In [0]:
# COMMAND ----------

# =============================================================================
# BASELINE RUNS + ZERO-DEMAND DIAGNOSTICS
# =============================================================================

# Zero vs. positive demand split — shows how error breaks down depending on whether
# actual demand existed. Shown before baseline comparison so you have a reference point
# for the XGBoost model's performance on these two populations.
pretzel_zero_vs_positive, nugget_zero_vs_positive, total_zero_vs_positive = mae_zero_vs_positive(eval_base)

display(pretzel_zero_vs_positive.orderBy("actual_bucket"))
display(nugget_zero_vs_positive.orderBy("actual_bucket"))
display(total_zero_vs_positive.orderBy("actual_bucket"))

# COMMAND ----------

# Metrics restricted to rows where actual sales were positive (> 0).
# This isolates "did we get the demand right when there was demand?"
# from the zero-demand noise. Often the most meaningful signal for the bakery.
positive_only_metrics = (
    eval_base
    .agg(
        F.mean(F.when(F.col("Sales of Pretzels") > 0, F.col("ae_p"))).alias("MAE_pretzels_actual_gt_0"),
        F.mean(F.when(F.col("Sales of Pretzel Nuggets") > 0, F.col("ae_n"))).alias("MAE_nuggets_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzels") > 0, F.col("ae_p"))).alias("sum_ae_p_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzel Nuggets") > 0, F.col("ae_n"))).alias("sum_ae_n_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzels") > 0, 1).otherwise(0)).alias("pretzel_positive_rows"),
        F.sum(F.when(F.col("Sales of Pretzel Nuggets") > 0, 1).otherwise(0)).alias("nugget_positive_rows")
    )
)
display(positive_only_metrics)

positive_only_weekly_metrics = (
    eval_base
    .groupBy("forecast_week_start")
    .agg(
        F.mean(F.when(F.col("Sales of Pretzels") > 0, F.col("ae_p"))).alias("MAE_pretzels_actual_gt_0"),
        F.mean(F.when(F.col("Sales of Pretzel Nuggets") > 0, F.col("ae_n"))).alias("MAE_nuggets_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzels") > 0, F.col("ae_p"))).alias("sum_ae_p_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzel Nuggets") > 0, F.col("ae_n"))).alias("sum_ae_n_actual_gt_0"),
        F.sum(F.when(F.col("Sales of Pretzels") > 0, 1).otherwise(0)).alias("pretzel_positive_rows"),
        F.sum(F.when(F.col("Sales of Pretzel Nuggets") > 0, 1).otherwise(0)).alias("nugget_positive_rows")
    )
    .orderBy("forecast_week_start")
)
display(positive_only_weekly_metrics)

# COMMAND ----------

# Run the baseline function for each eval week and stack the results.
# This gives us one row per store/hour/week for each baseline method,
# ready to be compared against the XGBoost model in Block 16.
baseline_week_outputs = []

for ws in eval_week_starts:
    baseline_week_outputs.append(run_one_week_baselines(full, ws))

baseline_pred = baseline_week_outputs[0]
for df in baseline_week_outputs[1:]:
    baseline_pred = baseline_pred.unionByName(df)

_ = baseline_pred.count()

compare_total_vs_distinct(baseline_pred, ["Store id", "Time by hour"], "baseline_pred")

display(
    baseline_pred.orderBy("forecast_week_start", "Store id", "Time by hour").limit(1000)
)


metric,actual_bucket,N,MAE_pretzels,sum_ae_p,pretzel_volume
pretzels,actual = 0,85614,0.38910224573085656,33312.59966600155,0.0
pretzels,actual > 0,102028,2.3679653130634857,241598.76496124134,564636.0


metric,actual_bucket,N,MAE_nuggets,sum_ae_n,nugget_volume
nuggets,actual = 0,68349,0.3680707796924093,25157.269721196484,0.0
nuggets,actual > 0,119293,4.684325786516143,558807.2760508703,1651390.0


metric,actual_bucket,N,MAE_total,sum_ae_total,total_volume
total,actual = 0,66502,0.4127022266160929,27445.52347442341,0.0
total,actual > 0,121140,5.772878156387923,699326.4598648329,2216026.0


MAE_pretzels_actual_gt_0,MAE_nuggets_actual_gt_0,sum_ae_p_actual_gt_0,sum_ae_n_actual_gt_0,pretzel_positive_rows,nugget_positive_rows
2.3679653130634857,4.684325786516142,241598.76496124134,558807.2760508702,102028,119293


forecast_week_start,MAE_pretzels_actual_gt_0,MAE_nuggets_actual_gt_0,sum_ae_p_actual_gt_0,sum_ae_n_actual_gt_0,pretzel_positive_rows,nugget_positive_rows
2026-04-06T00:00:00.000Z,2.4823501785673208,4.920053526248797,127945.29290371685,295050.6899156141,51542,59969
2026-04-13T00:00:00.000Z,2.251187894812908,4.446035097688219,113653.47205752449,263756.5861352559,50486,59324


...

dataset,total_rows,distinct_rows,duplicate_rows,duplicate_rate
baseline_week,93821,93821,0,0.0



baselines for week starting 2026-04-13 00:00:00


dataset,total_rows,distinct_rows,duplicate_rows,duplicate_rate
baseline_week,93821,93821,0,0.0


dataset,total_rows,distinct_rows,duplicate_rows,duplicate_rate
baseline_pred,187642,187642,0,0.0


Store id,Time by hour,forecast_week_start,hour,day_of_week,is_weekend,is_holiday,Sales of Pretzels,Sales of Pretzel Nuggets,bl_lastweek_p,bl_lastweek_n,bl_lastyear_p,bl_lastyear_n,bl_avg_1w_p,bl_avg_1w_n,bl_avg_2w_p,bl_avg_2w_n,bl_avg_3w_p,bl_avg_3w_n,bl_avg_4w_p,bl_avg_4w_n,bl_train_store_hour_avg_p,bl_train_store_hour_avg_n,bl_train_store_hour_dow_avg_p,bl_train_store_hour_dow_avg_n
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T10:00:00.000Z,2026-04-06T00:00:00.000Z,10,2,0,0,4.0,2.0,6.0,0.0,6.0,3.0,6.0,0.0,7.0,2.0,6.333333333333333,1.6666666666666667,4.75,1.25,7.5212121212121215,5.082424242424242,5.572649572649572,3.47008547008547
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T11:00:00.000Z,2026-04-06T00:00:00.000Z,11,2,0,0,12.0,10.0,11.0,5.0,8.0,3.0,11.0,5.0,10.5,6.0,9.333333333333334,5.0,7.0,3.75,10.743030303030302,8.118787878787879,8.606837606837606,6.760683760683761
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T12:00:00.000Z,2026-04-06T00:00:00.000Z,12,2,0,0,12.0,9.0,8.0,5.0,11.0,10.0,8.0,5.0,10.5,5.5,10.666666666666666,5.666666666666667,8.75,5.5,13.066666666666666,9.684848484848485,10.675213675213675,8.427350427350428
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T13:00:00.000Z,2026-04-06T00:00:00.000Z,13,2,0,0,10.0,9.0,14.0,5.0,12.0,8.0,14.0,5.0,12.5,6.5,10.0,7.0,9.0,7.25,13.026666666666667,9.975757575757576,10.17094017094017,7.957264957264957
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T14:00:00.000Z,2026-04-06T00:00:00.000Z,14,2,0,0,15.0,13.0,12.0,8.0,12.0,12.0,12.0,8.0,15.0,10.0,12.666666666666666,10.333333333333334,11.0,9.0,12.792727272727273,10.025454545454545,10.418803418803419,8.452991452991453
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T15:00:00.000Z,2026-04-06T00:00:00.000Z,15,2,0,0,11.0,16.0,10.0,7.0,9.0,9.0,10.0,7.0,10.0,7.5,10.666666666666666,8.666666666666666,10.75,7.75,12.61939393939394,10.167272727272728,10.94017094017094,8.478632478632479
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T16:00:00.000Z,2026-04-06T00:00:00.000Z,16,2,0,0,17.0,10.0,7.0,7.0,8.0,6.0,7.0,7.0,9.5,9.0,8.666666666666666,7.0,8.0,7.75,11.64,9.884848484848485,10.393162393162394,9.444444444444445
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T17:00:00.000Z,2026-04-06T00:00:00.000Z,17,2,0,0,8.0,9.0,4.0,6.0,11.0,17.0,4.0,6.0,5.5,5.5,6.0,5.0,7.25,5.5,9.304242424242425,8.243636363636364,8.401709401709402,8.64102564102564
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T18:00:00.000Z,2026-04-06T00:00:00.000Z,18,2,0,0,6.0,11.0,7.0,1.0,4.0,7.0,7.0,1.0,5.0,3.5,5.333333333333333,5.666666666666667,6.0,6.25,7.244848484848485,6.899393939393939,6.897435897435898,7.444444444444445
0011c73b3b99e4d76926c6ac0a40f8dfd9b5692efcfd365d2aa7fadeee1f526f,2026-04-06T19:00:00.000Z,2026-04-06T00:00:00.000Z,19,2,0,0,12.0,9.0,4.0,5.0,4.0,11.0,4.0,5.0,3.5,4.0,5.0,4.666666666666667,5.25,5.5,5.581818181818182,5.463030303030303,5.47008547008547,6.068376068376068


Block 15 — baseline long + eval

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark import StorageLevel

# =============================================================================
# BASELINE LONG FORMAT + EVAL
# =============================================================================
# Reshape the baseline results from wide format (one column per baseline method)
# into long format (one row per baseline method per store/hour).
# This makes it easy to compute metrics for all baselines simultaneously
# using the same error functions used for the XGBoost model.

baseline_long = (
    baseline_pred
    .selectExpr(
        "`Store id`",
        "`Time by hour`",
        "forecast_week_start",
        "hour",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        f"`{ACT_P}` as `{ACT_P}`",
        f"`{ACT_N}` as `{ACT_N}`",
        # stack() turns each baseline column pair into a separate row with a label
        """
        stack(
            8,
            'same_store_hour_last_week', bl_lastweek_p, bl_lastweek_n,
            'same_store_hour_same_weekday_last_year_364d', bl_lastyear_p, bl_lastyear_n,
            'avg_same_store_hour_last_1_week', bl_avg_1w_p, bl_avg_1w_n,
            'avg_same_store_hour_last_2_weeks', bl_avg_2w_p, bl_avg_2w_n,
            'avg_same_store_hour_last_3_weeks', bl_avg_3w_p, bl_avg_3w_n,
            'avg_same_store_hour_last_4_weeks', bl_avg_4w_p, bl_avg_4w_n,
            'train_store_hour_mean', bl_train_store_hour_avg_p, bl_train_store_hour_avg_n,
            'train_store_hour_dayofweek_mean', bl_train_store_hour_dow_avg_p, bl_train_store_hour_dow_avg_n
        ) as (baseline, pred_pretzels, pred_nuggets)
        """
    )
    # Drop rows where the baseline itself is null — some baselines won't exist for new stores
    .dropna(subset=["pred_pretzels", "pred_nuggets", ACT_P, ACT_N])
)

# Apply the same error column logic as the main model eval
baseline_eval = (
    add_error_cols(baseline_long)
    .select(
        "baseline", "forecast_week_start",
        ACT_P, ACT_N, "pred_pretzels", "pred_nuggets",
        "actual_total", "pred_total",
        "err_p", "err_n", "err_total",
        "ae_p", "ae_n", "ae_total",
        "se_p", "se_n", "se_total"
    )
)

_ = baseline_eval.count()


Block 16 — comparison tables

In [0]:
# COMMAND ----------

# =============================================================================
# COMPARISON TABLES — MODEL VS. BASELINES
# =============================================================================
# This is the key validation step: does the XGBoost model actually beat
# the naive baselines? If "last week's value" or "4-week average" performs
# comparably, the ML complexity may not be justified.
# Results are sorted by WAPE ascending — lower is better.

# Aggregate metrics for each baseline method across all eval weeks
baseline_metrics = (
    baseline_eval
    .groupBy("baseline")
    .agg(
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.mean(F.when(F.col(ACT_P) != 0, F.col("ae_p") / F.col(ACT_P))).alias("MAPE_pretzels"),
        F.mean(F.when(F.col(ACT_N) != 0, F.col("ae_n") / F.col(ACT_N))).alias("MAPE_nuggets"),
        F.mean(F.when(F.col("actual_total") != 0, F.col("ae_total") / F.col("actual_total"))).alias("MAPE_total"),
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume")
    )
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .select(
        "baseline", "N",
        "RMSE_pretzels", "RMSE_nuggets", "RMSE_total",
        "MAE_pretzels", "MAE_nuggets", "MAE_total",
        "Bias_pretzels", "Bias_nuggets", "Bias_total",
        "MAPE_pretzels", "MAPE_nuggets", "MAPE_total",
        "wape_pretzels", "wape_nuggets", "wape_total"
    )
)

# Same metrics for the XGBoost model, labelled "current_model" for easy comparison
model_summary = (
    eval_base
    .agg(
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.mean(F.when(F.col(ACT_P) != 0, F.col("ae_p") / F.col(ACT_P))).alias("MAPE_pretzels"),
        F.mean(F.when(F.col(ACT_N) != 0, F.col("ae_n") / F.col(ACT_N))).alias("MAPE_nuggets"),
        F.mean(F.when(F.col("actual_total") != 0, F.col("ae_total") / F.col("actual_total"))).alias("MAPE_total"),
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume")
    )
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .withColumn("baseline", F.lit("current_model"))
    .select(
        "baseline", "N",
        "RMSE_pretzels", "RMSE_nuggets", "RMSE_total",
        "MAE_pretzels", "MAE_nuggets", "MAE_total",
        "Bias_pretzels", "Bias_nuggets", "Bias_total",
        "MAPE_pretzels", "MAPE_nuggets", "MAPE_total",
        "wape_pretzels", "wape_nuggets", "wape_total"
    )
)

# Stack model and baseline rows together — one table, easy to compare
comparison_table = model_summary.unionByName(baseline_metrics)
display(comparison_table.orderBy("wape_total"))

# COMMAND ----------

# Same comparison but broken down by week — useful for checking if the model's
# advantage over baselines holds consistently or varies across weeks.
baseline_weekly_table = (
    baseline_eval
    .groupBy("baseline", "forecast_week_start")
    .agg(
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.mean(F.when(F.col(ACT_P) != 0, F.col("ae_p") / F.col(ACT_P))).alias("MAPE_pretzels"),
        F.mean(F.when(F.col(ACT_N) != 0, F.col("ae_n") / F.col(ACT_N))).alias("MAPE_nuggets"),
        F.mean(F.when(F.col("actual_total") != 0, F.col("ae_total") / F.col("actual_total"))).alias("MAPE_total"),
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume")
    )
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .select(
        "baseline", "forecast_week_start", "N",
        "RMSE_pretzels", "RMSE_nuggets", "RMSE_total",
        "MAE_pretzels", "MAE_nuggets", "MAE_total",
        "Bias_pretzels", "Bias_nuggets", "Bias_total",
        "MAPE_pretzels", "MAPE_nuggets", "MAPE_total",
        "wape_pretzels", "wape_nuggets", "wape_total"
    )
)

model_weekly_table = (
    eval_base
    .groupBy("forecast_week_start")
    .agg(
        F.count("*").alias("N"),
        F.sqrt(F.mean("se_p")).alias("RMSE_pretzels"),
        F.sqrt(F.mean("se_n")).alias("RMSE_nuggets"),
        F.sqrt(F.mean("se_total")).alias("RMSE_total"),
        F.mean("ae_p").alias("MAE_pretzels"),
        F.mean("ae_n").alias("MAE_nuggets"),
        F.mean("ae_total").alias("MAE_total"),
        F.mean("err_p").alias("Bias_pretzels"),
        F.mean("err_n").alias("Bias_nuggets"),
        F.mean("err_total").alias("Bias_total"),
        F.mean(F.when(F.col(ACT_P) != 0, F.col("ae_p") / F.col(ACT_P))).alias("MAPE_pretzels"),
        F.mean(F.when(F.col(ACT_N) != 0, F.col("ae_n") / F.col(ACT_N))).alias("MAPE_nuggets"),
        F.mean(F.when(F.col("actual_total") != 0, F.col("ae_total") / F.col("actual_total"))).alias("MAPE_total"),
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume")
    )
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .withColumn("baseline", F.lit("current_model"))
    .select(
        "baseline", "forecast_week_start", "N",
        "RMSE_pretzels", "RMSE_nuggets", "RMSE_total",
        "MAE_pretzels", "MAE_nuggets", "MAE_total",
        "Bias_pretzels", "Bias_nuggets", "Bias_total",
        "MAPE_pretzels", "MAPE_nuggets", "MAPE_total",
        "wape_pretzels", "wape_nuggets", "wape_total"
    )
)

weekly_comparison_table = model_weekly_table.unionByName(baseline_weekly_table)
display(weekly_comparison_table.orderBy("forecast_week_start", "wape_total"))


baseline,N,RMSE_pretzels,RMSE_nuggets,RMSE_total,MAE_pretzels,MAE_nuggets,MAE_total,Bias_pretzels,Bias_nuggets,Bias_total,MAPE_pretzels,MAPE_nuggets,MAPE_total,wape_pretzels,wape_nuggets,wape_total
current_model,187642,3.560200441278803,5.620198235411867,7.309553410018806,1.465084387435875,3.1121206647342654,3.8731839531621706,-0.20687605633625245,-0.046465322102729065,-0.2533413784389816,0.601032415977537,0.5606410438219114,0.5072893927611827,0.4868824598984876,0.35362000846079183,0.32796184852490723
avg_same_store_hour_last_4_weeks,187642,3.713578265969488,5.959984740977334,7.74204400303039,1.571533825049829,3.3339803988446084,4.170073864060285,0.03678947144029588,0.4531887850268064,0.48997825646710225,0.689031592503776,0.6224273475416529,0.5725316510277169,0.5222581450704524,0.37882919843283536,0.3531010015225453
avg_same_store_hour_last_3_weeks,187642,3.849365130298856,6.213823317585714,8.092232660092332,1.6320741980295745,3.4640840892053353,4.347086473177661,0.029415944546885422,0.35899745259590193,0.38841339714278694,0.7168541428620381,0.6405587699914885,0.5902573408623851,0.5423771538950145,0.39361245173258136,0.36808954407574757
train_store_hour_dayofweek_mean,187642,3.643754710281414,5.863434098262069,7.738550352351869,1.6618824363968063,3.424432247716536,4.412872475095103,0.30951070047573037,0.24396575235300602,0.5534764528287348,0.6787312089556585,0.5661971393763608,0.5434137541261825,0.552283141936344,0.38910694374195454,0.373659973742093
avg_same_store_hour_last_2_weeks,187642,4.165424311078897,7.11569770471151,9.243601874323879,1.7717168864113577,3.9092926956651497,4.923721768047665,0.011796399526758402,0.22313501241726266,0.23493141194402106,0.7792190040139925,0.703587287807765,0.6488934535616138,0.5887837474054081,0.4442000375441295,0.4169161372655375
same_store_hour_last_week,187642,5.033157132857897,8.534197179800483,11.139319670221496,2.038088487652018,4.489964933223905,5.643859050745569,0.05369267008452266,0.346734739557242,0.40042740964176465,0.9108587789160243,0.8014436981056641,0.7384482857843813,0.6773053790406562,0.5101799090463186,0.47789376117428223
avg_same_store_hour_last_1_week,187642,5.033157132857897,8.534197179800483,11.139319670221496,2.038088487652018,4.489964933223905,5.643859050745569,0.05369267008452266,0.346734739557242,0.40042740964176465,0.9108587789160243,0.8014436981056641,0.7384482857843813,0.6773053790406562,0.5101799090463186,0.47789376117428223
same_store_hour_same_weekday_last_year_364d,187642,5.149337704267841,9.327805564056176,12.218696905823448,2.304201617974654,5.115789641977809,6.577152236706068,0.06792189381908102,0.09878918365824282,0.16671107747732383,0.9584951465727355,0.8341155583391114,0.7836618356238749,0.765741114629602,0.5812903069535361,0.5569203610426954
train_store_hour_mean,187642,4.097021382983351,8.459383097252688,10.925859007767778,2.00052825727584,5.159990608448998,6.6576554153058325,0.31029416076536964,0.24635166945143988,0.5566458302168168,0.7943705549495033,0.8244675443068399,0.7957653712546947,0.6648232192983676,0.5863127170145072,0.5637369676343225


baseline,forecast_week_start,N,RMSE_pretzels,RMSE_nuggets,RMSE_total,MAE_pretzels,MAE_nuggets,MAE_total,Bias_pretzels,Bias_nuggets,Bias_total,MAPE_pretzels,MAPE_nuggets,MAPE_total,wape_pretzels,wape_nuggets,wape_total
current_model,2026-04-06T00:00:00.000Z,93821,4.329012411133449,6.01068293844229,8.115887738748029,1.529522749777456,3.2696769565737096,4.097245202991963,-0.4077888971736428,-0.8311496694530283,-1.2389385666266715,0.5693141286108927,0.4859997806687333,0.4478062002800779,0.47924041580600363,0.3424901994251369,0.32164692855037424
avg_same_store_hour_last_4_weeks,2026-04-06T00:00:00.000Z,93821,4.388125916050677,6.154957749025975,8.266251511684288,1.6110518966969016,3.390165847731318,4.244148431587811,-0.1680380724997602,-0.32837797508020594,-0.4964160475799661,0.6427772855030045,0.5276198438126817,0.49197435591801875,0.5047856796967622,0.3551110989541001,0.33317930455693684
avg_same_store_hour_last_3_weeks,2026-04-06T00:00:00.000Z,93821,4.464644289350478,6.38486731200756,8.542173198104482,1.666425071856693,3.5224096950576067,4.415958758344791,-0.16435908094492002,-0.34306463016453315,-0.5074237111094531,0.6703169730945068,0.5491131880188225,0.5121702323453811,0.5221355775599607,0.36896329972043806,0.34666696789089213
train_store_hour_dayofweek_mean,2026-04-06T00:00:00.000Z,93821,4.3457153442867416,6.226122949448792,8.398304447853883,1.6924323652845086,3.5528147735464106,4.534623663902209,0.12760278405207387,-0.5041876464610054,-0.37658486240893163,0.6356208106944483,0.49027479044597017,0.47618947248005444,0.5302843586867196,0.37214815300517345,0.35598254470123086
avg_same_store_hour_last_2_weeks,2026-04-06T00:00:00.000Z,93821,4.638598433980503,6.99624697829993,9.292929093621881,1.78008121849053,3.826248920817301,4.820578548512593,-0.2125430340755268,-0.6464917236013259,-0.8590347576768528,0.7159176354593396,0.577915434638282,0.5402741014592961,0.5577470903534991,0.4007896722966033,0.3784309230095982
same_store_hour_same_weekday_last_year_364d,2026-04-06T00:00:00.000Z,93821,5.202707152261852,7.79703440653647,10.3824634282618,2.180066296458149,4.480553394229437,5.72319629933597,-0.08737915818420183,-0.6524552072563712,-0.739834365440573,0.8549498318306643,0.6471874778361627,0.6190602439301371,0.683073121044634,0.46932637257616494,0.44928932001141303
train_store_hour_mean,2026-04-06T00:00:00.000Z,93821,4.667828502027337,8.233834620914887,10.826487000274637,1.9685183875290506,4.935591405038557,6.337577223940001,0.12839110453077462,-0.5017862189540178,-0.3733951144232427,0.7218270836879482,0.6722450501234848,0.6615100555066039,0.6167894990110143,0.5169904265906459,0.49752019894795335
same_store_hour_last_week,2026-04-06T00:00:00.000Z,93821,5.290317082576957,9.701039492412583,12.650435616210379,2.1893499323179246,5.04556549173426,6.385595975314695,-0.2575009859199966,-0.7985845386427346,-1.0560855245627312,0.8858690801639031,0.729709437405494,0.6842106072243943,0.685981932639805,0.5285099275640625,0.5012898253987247
avg_same_store_hour_last_1_week,2026-04-06T00:00:00.000Z,93821,5.290317082576957,9.701039492412583,12.650435616210379,2.1893499323179246,5.04556549173426,6.385595975314695,-0.2575009859199966,-0.7985845386427346,-1.0560855245627312,0.8858690801639031,0.729709437405494,0.6842106072243943,0.685981932639805,0.5285099275640625,0.5012898253987247
current_model,2026-04-13T00:00:00.000Z,93821,2.5709348316159493,5.200475653646702,6.402461114280546,1.4006460250943003,2.9545643728948168,3.649122703332383,-0.005963215498862182,0.73821902524757,0.7322558097487085,0.6334141449212363,0.6360938438696653,0.5672240743153281,0.49551099249389086,0.36681149980993116,0.33535442755026434


Block 17 — remaining diagnostics

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F

# =============================================================================
# REMAINING DIAGNOSTICS
# =============================================================================

# --- Daily and weekly aggregate accuracy ---
# Instead of looking at individual store/hour accuracy, collapse everything to
# daily and weekly totals. This is the "network level" view — across all stores,
# did we predict the right total volume? This matters for supply chain decisions
# where individual store variance averages out.

daily_total_capture = (
    pred_final
    .withColumn("forecast_date", F.to_date("Time by hour"))
    .groupBy("forecast_date")
    .agg(
        F.sum(F.col(ACT_P)).alias("actual_pretzels"),
        F.sum(F.col(ACT_N)).alias("actual_nuggets"),
        F.sum(F.col("pred_pretzels")).alias("pred_pretzels"),
        F.sum(F.col("pred_nuggets")).alias("pred_nuggets")
    )
    .withColumn("actual_total", F.col("actual_pretzels") + F.col("actual_nuggets"))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    .withColumn("ae_p", F.abs(F.col("pred_pretzels") - F.col("actual_pretzels")))
    .withColumn("ae_n", F.abs(F.col("pred_nuggets") - F.col("actual_nuggets")))
    .withColumn("ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    .withColumn("wape_pretzels", F.expr("try_divide(ae_p, actual_pretzels)"))
    .withColumn("wape_nuggets", F.expr("try_divide(ae_n, actual_nuggets)"))
    .withColumn("wape_total", F.expr("try_divide(ae_total, actual_total)"))
    .orderBy("forecast_date")
)

weekly_total_capture = (
    pred_final
    .groupBy("forecast_week_start")
    .agg(
        F.sum(F.col(ACT_P)).alias("actual_pretzels"),
        F.sum(F.col(ACT_N)).alias("actual_nuggets"),
        F.sum(F.col("pred_pretzels")).alias("pred_pretzels"),
        F.sum(F.col("pred_nuggets")).alias("pred_nuggets")
    )
    .withColumn("actual_total", F.col("actual_pretzels") + F.col("actual_nuggets"))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    .withColumn("ae_p", F.abs(F.col("pred_pretzels") - F.col("actual_pretzels")))
    .withColumn("ae_n", F.abs(F.col("pred_nuggets") - F.col("actual_nuggets")))
    .withColumn("ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    .withColumn("wape_pretzels", F.expr("try_divide(ae_p, actual_pretzels)"))
    .withColumn("wape_nuggets", F.expr("try_divide(ae_n, actual_nuggets)"))
    .withColumn("wape_total", F.expr("try_divide(ae_total, actual_total)"))
    .orderBy("forecast_week_start")
)

display(daily_total_capture)
display(weekly_total_capture)

# COMMAND ----------

# Single-number summary of daily and weekly level WAPE.
# Daily WAPE is harder to achieve than weekly because there's less averaging out —
# a single bad day drags it down. Weekly WAPE is the more forgiving headline number.
daily_total_capture_summary = (
    daily_total_capture
    .agg(
        F.expr("try_divide(sum(ae_p), sum(actual_pretzels))").alias("daily_level_wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(actual_nuggets))").alias("daily_level_wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("daily_level_wape_total")
    )
)

weekly_total_capture_summary = (
    weekly_total_capture
    .agg(
        F.expr("try_divide(sum(ae_p), sum(actual_pretzels))").alias("weekly_level_wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(actual_nuggets))").alias("weekly_level_wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("weekly_level_wape_total")
    )
)

display(daily_total_capture_summary)
display(weekly_total_capture_summary)

# COMMAND ----------

# Zero confusion matrix — a 2×2 breakdown of:
#   actual=0 & pred=0  → correctly predicted nothing needed (good)
#   actual=0 & pred>0  → false alarm (wasted baking)
#   actual>0 & pred=0  → missed demand (ran out)
#   actual>0 & pred>0  → correctly predicted some demand (good)
# The two bad quadrants tell you the character of the model's failures.
# COMMAND ----------

zero_diagnostics = (
    pred_final
    .select(
        F.sum(((F.col("Sales of Pretzels") == 0) & (F.col("pred_pretzels") == 0)).cast("int")).alias("pretzel_actual0_pred0"),
        F.sum(((F.col("Sales of Pretzels") == 0) & (F.col("pred_pretzels") > 0)).cast("int")).alias("pretzel_actual0_predpos"),
        F.sum(((F.col("Sales of Pretzels") > 0) & (F.col("pred_pretzels") == 0)).cast("int")).alias("pretzel_actualpos_pred0"),
        F.sum(((F.col("Sales of Pretzels") > 0) & (F.col("pred_pretzels") > 0)).cast("int")).alias("pretzel_actualpos_predpos"),

        F.sum(((F.col("Sales of Pretzel Nuggets") == 0) & (F.col("pred_nuggets") == 0)).cast("int")).alias("nugget_actual0_pred0"),
        F.sum(((F.col("Sales of Pretzel Nuggets") == 0) & (F.col("pred_nuggets") > 0)).cast("int")).alias("nugget_actual0_predpos"),
        F.sum(((F.col("Sales of Pretzel Nuggets") > 0) & (F.col("pred_nuggets") == 0)).cast("int")).alias("nugget_actualpos_pred0"),
        F.sum(((F.col("Sales of Pretzel Nuggets") > 0) & (F.col("pred_nuggets") > 0)).cast("int")).alias("nugget_actualpos_predpos")
    )
)

display(zero_diagnostics)

baseline_vs_model = (
    pred_final
    .withColumn("actual_total", F.col(ACT_P) + F.col(ACT_N))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    .withColumn("baseline_total", F.col("baseline_p") + F.col("baseline_n"))
    .withColumn("baseline_ae_total", F.abs(F.col("baseline_total") - F.col("actual_total")))
    .withColumn("model_ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    .withColumn("model_beats_baseline", (F.col("model_ae_total") < F.col("baseline_ae_total")).cast("double"))
    .agg(
        F.mean("model_beats_baseline").alias("pct_rows_model_beats_baseline"),
        F.mean(F.col("model_ae_total") - F.col("baseline_ae_total")).alias("mean_ae_change_vs_baseline")
    )
)

display(baseline_vs_model)



forecast_date,actual_pretzels,actual_nuggets,pred_pretzels,pred_nuggets,actual_total,pred_total,ae_p,ae_n,ae_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06,38140.0,115603.0,27672.244072030684,81434.38095944856,153743.0,109106.62503147926,10467.755927969316,34168.619040551435,44636.374968520744,0.2744561071832542,0.29556861881224045,0.29033110430081854
2026-04-07,34009.0,93796.0,28692.709491491296,79956.63621216347,127805.0,108649.34570365478,5316.290508508704,13839.363787836526,19155.654296345223,0.15632010669260207,0.14754748377155238,0.14988188487418508
2026-04-08,33929.0,95592.0,28965.947667315562,83997.0708130882,129521.0,112963.01848040376,4963.052332684438,11594.929186911802,16557.981519596244,0.14627758945693767,0.12129602045057958,0.12784013032323904
2026-04-09,35337.0,98670.0,30896.62268250561,91944.220683779,134007.0,122840.84336628461,4440.377317494389,6725.779316221,11166.156633715393,0.1256580161726912,0.06816437940834094,0.08332517430966586
2026-04-10,44379.0,128793.0,43485.670179359826,139456.26808074964,173172.0,182941.93826010945,893.3298206401741,10663.268080749636,9769.938260109455,0.020129561744072065,0.08279384811868375,0.05641754013414094
2026-04-11,66656.0,214809.0,63986.080866037024,219448.4957716454,281465.0,283434.5766376824,2669.919133962976,4639.495771645394,1969.5766376823885,0.04005519584077916,0.021598237372016042,0.006997589887490056
2026-04-12,46985.0,148425.0,37476.56291953166,121471.63434137318,195410.0,158948.19726090482,9508.437080468342,26953.36565862682,36461.802739095176,0.2023717586563444,0.18159586093061694,0.18659128365536654
2026-04-13,26938.0,68054.0,29135.61156721333,88218.53673946999,94992.0,117354.14830668332,2197.611567213331,20164.536739469986,22362.148306683317,0.08158035367188844,0.29630200634011206,0.2354108588795195
2026-04-14,27112.0,65276.0,29498.973205327035,82840.19373014983,92388.0,112339.16693547685,2386.9732053270345,17564.193730149826,19951.166935476853,0.08804120704215973,0.26907582771845434,0.21594976550500988
2026-04-15,27163.0,69887.0,29635.29424896813,86281.2753195219,97050.0,115916.56956849003,2472.29424896813,16394.275319521897,18866.569568490027,0.09101698078150904,0.23458261650266712,0.19440051075208684


forecast_week_start,actual_pretzels,actual_nuggets,pred_pretzels,pred_nuggets,actual_total,pred_total,ae_p,ae_n,ae_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06T00:00:00.000Z,299435.0,895688.0,261175.8378782716,817708.7068622478,1195123.0,1078884.5447405195,38259.16212172841,77979.29313775222,116238.45525948051,0.12777117612078887,0.0870607768974824,0.09726066292714684
2026-04-13T00:00:00.000Z,265201.0,755702.0,264641.5251586812,824962.4471677522,1020903.0,1089603.9723264333,559.4748413187917,69260.4471677522,68700.97232643329,0.002109625685117295,0.09165047487998206,0.06729431917276497


daily_level_wape_pretzels,daily_level_wape_nuggets,daily_level_wape_total
0.11621847309927934,0.14422622190523382,0.13233308937359625


weekly_level_wape_pretzels,weekly_level_wape_nuggets,weekly_level_wape_total
0.06874984408193498,0.08916109477803814,0.0834554412204162


pretzel_actual0_pred0,pretzel_actual0_predpos,pretzel_actualpos_pred0,pretzel_actualpos_predpos,nugget_actual0_pred0,nugget_actual0_predpos,nugget_actualpos_pred0,nugget_actualpos_predpos
63680,21934,1400,100628,58630,9719,964,118329


pct_rows_model_beats_baseline,mean_ae_change_vs_baseline
0.4217925624327176,-0.2968899108981125


Block 18 — signed error distribution and exact / within-k

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import Window

# Calculate signed prediction errors for both pretzels and nuggets
# This transformation rounds predictions to integers and computes the difference (pred - actual)
# Positive errors = overestimation, negative errors = underestimation
signed_error_dist = (
    pred_final
    .select(
        # Cast actual sales values to integers for comparison
        F.col("Sales of Pretzels").cast("int").alias("actual_p"),
        F.col("Sales of Pretzel Nuggets").cast("int").alias("actual_n"),
        # Round predicted values to nearest integer for fair comparison
        F.round(F.col("pred_pretzels")).cast("int").alias("pred_p_round"),
        F.round(F.col("pred_nuggets")).cast("int").alias("pred_n_round")
    )
    # Compute signed error: positive means we over-predicted, negative means we under-predicted
    .withColumn("signed_err_p", F.col("pred_p_round") - F.col("actual_p"))
    .withColumn("signed_err_n", F.col("pred_n_round") - F.col("actual_n"))
)

# Aggregate pretzel predictions by their signed error values
# This shows the distribution: how many predictions were off by -5, -4, ..., 0, ..., +4, +5, etc.
pretzel_signed_dist = (
    signed_error_dist
    .groupBy("signed_err_p")  # Group by each unique error value
    .agg(F.count("*").alias("N"))  # Count how many predictions had this error
    # Calculate what percentage of total rows each error value represents
    .withColumn("pct_of_rows", F.col("N") / F.sum("N").over(Window.partitionBy()))
    .orderBy("signed_err_p")  # Sort from most negative to most positive errors
)

# Same distribution analysis for nugget predictions
nugget_signed_dist = (
    signed_error_dist
    .groupBy("signed_err_n")
    .agg(F.count("*").alias("N"))
    .withColumn("pct_of_rows", F.col("N") / F.sum("N").over(Window.partitionBy()))
    .orderBy("signed_err_n")
)

display(pretzel_signed_dist)
display(nugget_signed_dist)

# COMMAND ----------

# Create a detailed breakdown of prediction accuracy by specific error buckets
# This computes the proportion of predictions falling into each discrete error category
error_bucket_summary = (
    pred_final
    .select(
        F.col("Sales of Pretzels").cast("int").alias("actual_p"),
        F.col("Sales of Pretzel Nuggets").cast("int").alias("actual_n"),
        F.round(F.col("pred_pretzels")).cast("int").alias("pred_p_round"),
        F.round(F.col("pred_nuggets")).cast("int").alias("pred_n_round")
    )
    # Calculate signed errors again (pred - actual)
    .withColumn("signed_err_p", F.col("pred_p_round") - F.col("actual_p"))
    .withColumn("signed_err_n", F.col("pred_n_round") - F.col("actual_n"))
    .agg(
        # Pretzel error buckets: each line calculates proportion of predictions with that exact error
        F.mean((F.col("signed_err_p") == 0).cast("double")).alias("pretzel_exact"),  # Perfect predictions
        F.mean((F.col("signed_err_p") == 1).cast("double")).alias("pretzel_plus_1"),  # Over by 1
        F.mean((F.col("signed_err_p") == -1).cast("double")).alias("pretzel_minus_1"),  # Under by 1
        F.mean((F.col("signed_err_p") == 2).cast("double")).alias("pretzel_plus_2"),  # Over by 2
        F.mean((F.col("signed_err_p") == -2).cast("double")).alias("pretzel_minus_2"),  # Under by 2
        F.mean((F.col("signed_err_p") == 3).cast("double")).alias("pretzel_plus_3"),  # Over by 3
        F.mean((F.col("signed_err_p") == -3).cast("double")).alias("pretzel_minus_3"),  # Under by 3
        F.mean((F.col("signed_err_p") == 4).cast("double")).alias("pretzel_plus_4"),  # Over by 4
        F.mean((F.col("signed_err_p") == -4).cast("double")).alias("pretzel_minus_4"),  # Under by 4
        F.mean((F.col("signed_err_p") == 5).cast("double")).alias("pretzel_plus_5"),  # Over by 5
        F.mean((F.col("signed_err_p") == -5).cast("double")).alias("pretzel_minus_5"),  # Under by 5
        # Catch-all for extreme errors (absolute error greater than 5)
        F.mean((F.abs(F.col("signed_err_p")) > 5).cast("double")).alias("pretzel_abs_gt_5"),

        # Nugget error buckets: identical structure to pretzel buckets
        F.mean((F.col("signed_err_n") == 0).cast("double")).alias("nugget_exact"),
        F.mean((F.col("signed_err_n") == 1).cast("double")).alias("nugget_plus_1"),
        F.mean((F.col("signed_err_n") == -1).cast("double")).alias("nugget_minus_1"),
        F.mean((F.col("signed_err_n") == 2).cast("double")).alias("nugget_plus_2"),
        F.mean((F.col("signed_err_n") == -2).cast("double")).alias("nugget_minus_2"),
        F.mean((F.col("signed_err_n") == 3).cast("double")).alias("nugget_plus_3"),
        F.mean((F.col("signed_err_n") == -3).cast("double")).alias("nugget_minus_3"),
        F.mean((F.col("signed_err_n") == 4).cast("double")).alias("nugget_plus_4"),
        F.mean((F.col("signed_err_n") == -4).cast("double")).alias("nugget_minus_4"),
        F.mean((F.col("signed_err_n") == 5).cast("double")).alias("nugget_plus_5"),
        F.mean((F.col("signed_err_n") == -5).cast("double")).alias("nugget_minus_5"),
        F.mean((F.abs(F.col("signed_err_n")) > 5).cast("double")).alias("nugget_abs_gt_5")
    )
)

display(error_bucket_summary)

# COMMAND ----------

# Calculate cumulative accuracy metrics showing % of predictions within increasing error tolerances
# Unlike the previous block which counted exact errors, this shows cumulative ranges
cumulative_accuracy = (
    pred_final
    .select(
        F.col("Sales of Pretzels").cast("int").alias("actual_p"),
        F.col("Sales of Pretzel Nuggets").cast("int").alias("actual_n"),
        F.round(F.col("pred_pretzels")).cast("int").alias("pred_p_round"),
        F.round(F.col("pred_nuggets")).cast("int").alias("pred_n_round")
    )
    # Use absolute error for cumulative metrics (we don't care about direction here)
    .withColumn("abs_err_p", F.abs(F.col("pred_p_round") - F.col("actual_p")))
    .withColumn("abs_err_n", F.abs(F.col("pred_n_round") - F.col("actual_n")))
    .agg(
        # Pretzel cumulative metrics: each line is inclusive of all smaller errors
        F.mean((F.col("abs_err_p") == 0).cast("double")).alias("pretzel_exact"),  # Exactly correct
        F.mean((F.col("abs_err_p") <= 1).cast("double")).alias("pretzel_within_1"),  # Off by 0 or 1
        F.mean((F.col("abs_err_p") <= 2).cast("double")).alias("pretzel_within_2"),  # Off by 0, 1, or 2
        F.mean((F.col("abs_err_p") <= 3).cast("double")).alias("pretzel_within_3"),  # Off by 0-3
        F.mean((F.col("abs_err_p") <= 5).cast("double")).alias("pretzel_within_5"),  # Off by 0-5
        F.mean((F.col("abs_err_p") <= 10).cast("double")).alias("pretzel_within_10"),  # Off by 0-10

        # Nugget cumulative metrics: same structure as pretzels
        F.mean((F.col("abs_err_n") == 0).cast("double")).alias("nugget_exact"),
        F.mean((F.col("abs_err_n") <= 1).cast("double")).alias("nugget_within_1"),
        F.mean((F.col("abs_err_n") <= 2).cast("double")).alias("nugget_within_2"),
        F.mean((F.col("abs_err_n") <= 3).cast("double")).alias("nugget_within_3"),
        F.mean((F.col("abs_err_n") <= 5).cast("double")).alias("nugget_within_5"),
        F.mean((F.col("abs_err_n") <= 10).cast("double")).alias("nugget_within_10")
    )
)

display(cumulative_accuracy)

# COMMAND ----------

# Break down cumulative accuracy metrics by forecast week
# This shows how model performance varies over time (e.g., does accuracy degrade for distant forecasts?)
weekly_cumulative_accuracy = (
    pred_final
    .select(
        "forecast_week_start",  # Keep the week dimension for time-based grouping
        F.col("Sales of Pretzels").cast("int").alias("actual_p"),
        F.col("Sales of Pretzel Nuggets").cast("int").alias("actual_n"),
        F.round(F.col("pred_pretzels")).cast("int").alias("pred_p_round"),
        F.round(F.col("pred_nuggets")).cast("int").alias("pred_n_round")
    )
    # Calculate absolute errors (direction doesn't matter for "within X" metrics)
    .withColumn("abs_err_p", F.abs(F.col("pred_p_round") - F.col("actual_p")))
    .withColumn("abs_err_n", F.abs(F.col("pred_n_round") - F.col("actual_n")))
    .groupBy("forecast_week_start")  # Compute metrics separately for each week
    .agg(
        # Pretzel accuracy by week: fewer tolerance levels than overall summary (0-3 instead of 0-10)
        F.mean((F.col("abs_err_p") == 0).cast("double")).alias("pretzel_exact"),
        F.mean((F.col("abs_err_p") <= 1).cast("double")).alias("pretzel_within_1"),
        F.mean((F.col("abs_err_p") <= 2).cast("double")).alias("pretzel_within_2"),
        F.mean((F.col("abs_err_p") <= 3).cast("double")).alias("pretzel_within_3"),
        # Nugget accuracy by week: same tolerance levels as pretzels
        F.mean((F.col("abs_err_n") == 0).cast("double")).alias("nugget_exact"),
        F.mean((F.col("abs_err_n") <= 1).cast("double")).alias("nugget_within_1"),
        F.mean((F.col("abs_err_n") <= 2).cast("double")).alias("nugget_within_2"),
        F.mean((F.col("abs_err_n") <= 3).cast("double")).alias("nugget_within_3")
    )
    .orderBy("forecast_week_start")  # Sort chronologically for trend analysis
)

display(weekly_cumulative_accuracy)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


signed_err_p,N,pct_of_rows
-947,1,5.3292972788608095E-6
-274,1,5.3292972788608095E-6
-125,1,5.3292972788608095E-6
-119,1,5.3292972788608095E-6
-113,1,5.3292972788608095E-6
-102,1,5.3292972788608095E-6
-89,1,5.3292972788608095E-6
-72,1,5.3292972788608095E-6
-66,1,5.3292972788608095E-6
-65,1,5.3292972788608095E-6


signed_err_n,N,pct_of_rows
-88,1,5.3292972788608095E-6
-87,1,5.3292972788608095E-6
-85,1,5.3292972788608095E-6
-84,1,5.3292972788608095E-6
-80,1,5.3292972788608095E-6
-79,1,5.3292972788608095E-6
-73,1,5.3292972788608095E-6
-72,1,5.3292972788608095E-6
-71,1,5.3292972788608095E-6
-68,1,5.3292972788608095E-6


pretzel_exact,pretzel_plus_1,pretzel_minus_1,pretzel_plus_2,pretzel_minus_2,pretzel_plus_3,pretzel_minus_3,pretzel_plus_4,pretzel_minus_4,pretzel_plus_5,pretzel_minus_5,pretzel_abs_gt_5,nugget_exact,nugget_plus_1,nugget_minus_1,nugget_plus_2,nugget_minus_2,nugget_plus_3,nugget_minus_3,nugget_plus_4,nugget_minus_4,nugget_plus_5,nugget_minus_5,nugget_abs_gt_5
0.460909604459556,0.12519052237771927,0.08490636424681042,0.07118342375374383,0.05845173255454536,0.037416996194881744,0.038754649811875805,0.01941462998688993,0.025969665639888725,0.010296202342759084,0.01632363756515066,0.05118257106617921,0.38585178158408034,0.07288346958570042,0.054993018620564695,0.055515289753893055,0.04377484784856269,0.04506986708732587,0.03411283188198804,0.03616461133434945,0.027968152119461527,0.028628984982040268,0.02153569030387653,0.19350145489815712


pretzel_exact,pretzel_within_1,pretzel_within_2,pretzel_within_3,pretzel_within_5,pretzel_within_10,nugget_exact,nugget_within_1,nugget_within_2,nugget_within_3,nugget_within_5,nugget_within_10
0.460909604459556,0.6710064910840856,0.8006416473923749,0.8768132933991324,0.9488174289338208,0.991574381002121,0.38585178158408034,0.5137282697903455,0.6130184073928012,0.6922011063621151,0.8064985451018428,0.9341512028223958


forecast_week_start,pretzel_exact,pretzel_within_1,pretzel_within_2,pretzel_within_3,nugget_exact,nugget_within_1,nugget_within_2,nugget_within_3
2026-04-06T00:00:00.000Z,0.4592468637085514,0.6663326973705247,0.7930420694727194,0.8687607252107737,0.38478592212830814,0.5111861949883288,0.6080941367071337,0.6854968503853082
2026-04-13T00:00:00.000Z,0.4625723452105605,0.6756802847976466,0.8082412253120304,0.884865861587491,0.3869176410398525,0.5162703445923621,0.6179426780784686,0.698905362338922


In [0]:
# MAGIC %md
# MAGIC Block 17 — remaining diagnostics

# COMMAND ----------

from pyspark.sql import functions as F

# Aggregate predictions and actuals to the daily level to measure total volume capture accuracy
# This rolls up hourly forecasts to see if we're hitting daily totals correctly
daily_total_capture = (
    pred_final
    .withColumn("forecast_date", F.to_date("Time by hour"))  # Extract date from hourly timestamp
    .groupBy("forecast_date")  # Aggregate all hours within each day
    .agg(
        # Sum up all actual sales across all hours/stores for this day
        F.sum(F.col(ACT_P)).alias("actual_pretzels"),
        F.sum(F.col(ACT_N)).alias("actual_nuggets"),
        # Sum up all predicted sales across all hours/stores for this day
        F.sum(F.col("pred_pretzels")).alias("pred_pretzels"),
        F.sum(F.col("pred_nuggets")).alias("pred_nuggets")
    )
    # Calculate combined totals (pretzels + nuggets) for both actual and predicted
    .withColumn("actual_total", F.col("actual_pretzels") + F.col("actual_nuggets"))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    # Calculate absolute errors at the daily level for each product and combined
    .withColumn("ae_p", F.abs(F.col("pred_pretzels") - F.col("actual_pretzels")))
    .withColumn("ae_n", F.abs(F.col("pred_nuggets") - F.col("actual_nuggets")))
    .withColumn("ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    # Calculate WAPE (Weighted Absolute Percentage Error) for each category
    # try_divide safely handles division by zero (returns null instead of error)
    .withColumn("wape_pretzels", F.expr("try_divide(ae_p, actual_pretzels)"))
    .withColumn("wape_nuggets", F.expr("try_divide(ae_n, actual_nuggets)"))
    .withColumn("wape_total", F.expr("try_divide(ae_total, actual_total)"))
    .orderBy("forecast_date")  # Sort chronologically
)

# Same aggregation as daily but at the weekly level
# Useful for smoothing out daily volatility and assessing longer-term capture
weekly_total_capture = (
    pred_final
    .groupBy("forecast_week_start")  # Aggregate all hours/days within each week
    .agg(
        # Sum all actual sales for the week
        F.sum(F.col(ACT_P)).alias("actual_pretzels"),
        F.sum(F.col(ACT_N)).alias("actual_nuggets"),
        # Sum all predictions for the week
        F.sum(F.col("pred_pretzels")).alias("pred_pretzels"),
        F.sum(F.col("pred_nuggets")).alias("pred_nuggets")
    )
    # Calculate weekly combined totals
    .withColumn("actual_total", F.col("actual_pretzels") + F.col("actual_nuggets"))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    # Calculate weekly-level absolute errors
    .withColumn("ae_p", F.abs(F.col("pred_pretzels") - F.col("actual_pretzels")))
    .withColumn("ae_n", F.abs(F.col("pred_nuggets") - F.col("actual_nuggets")))
    .withColumn("ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    # Calculate weekly-level WAPE metrics
    .withColumn("wape_pretzels", F.expr("try_divide(ae_p, actual_pretzels)"))
    .withColumn("wape_nuggets", F.expr("try_divide(ae_n, actual_nuggets)"))
    .withColumn("wape_total", F.expr("try_divide(ae_total, actual_total)"))
    .orderBy("forecast_week_start")  # Sort chronologically
)

display(daily_total_capture)
display(weekly_total_capture)

# COMMAND ----------

# Analyze model performance broken down by store volume segments
# This shows whether the model performs differently for high-volume vs low-volume stores
segment_perf = (
    add_error_cols(pred_final)  # Apply error calculation function (defined elsewhere)
    .groupBy("store_volume_segment")  # Group by volume tier (e.g., small/medium/large stores)
    .agg(
        F.count("*").alias("N"),  # Number of observations in this segment
        # Sum absolute errors across all observations in segment
        F.sum("ae_p").alias("sum_ae_p"),
        F.sum("ae_n").alias("sum_ae_n"),
        F.sum("ae_total").alias("sum_ae_total"),
        # Sum actual volumes to use as denominators for WAPE calculation
        F.sum(F.col(ACT_P)).alias("pretzel_volume"),
        F.sum(F.col(ACT_N)).alias("nugget_volume"),
        F.sum(F.col("actual_total")).alias("total_volume"),
        # Calculate mean signed error (bias) to detect systematic over/under-prediction
        F.mean("err_p").alias("bias_pretzels"),
        F.mean("err_n").alias("bias_nuggets"),
        F.mean("err_total").alias("bias_total")
    )
    # Calculate segment-level WAPE by dividing summed errors by summed actuals
    .withColumn("wape_pretzels", F.expr("try_divide(sum_ae_p, pretzel_volume)"))
    .withColumn("wape_nuggets", F.expr("try_divide(sum_ae_n, nugget_volume)"))
    .withColumn("wape_total", F.expr("try_divide(sum_ae_total, total_volume)"))
    .orderBy("store_volume_segment")  # Sort by segment for readable output
)

display(segment_perf)

# COMMAND ----------

# Calculate overall WAPE at the daily aggregation level
# This is the "top-line" accuracy metric for day-level forecasts across the entire test set
daily_total_capture_summary = (
    daily_total_capture
    .agg(
        # Global WAPE: sum of all daily errors / sum of all daily actuals
        F.expr("try_divide(sum(ae_p), sum(actual_pretzels))").alias("daily_level_wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(actual_nuggets))").alias("daily_level_wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("daily_level_wape_total")
    )
)

# Calculate overall WAPE at the weekly aggregation level
# Shows how well we capture total volume when rolled up to weeks
weekly_total_capture_summary = (
    weekly_total_capture
    .agg(
        # Global WAPE: sum of all weekly errors / sum of all weekly actuals
        F.expr("try_divide(sum(ae_p), sum(actual_pretzels))").alias("weekly_level_wape_pretzels"),
        F.expr("try_divide(sum(ae_n), sum(actual_nuggets))").alias("weekly_level_wape_nuggets"),
        F.expr("try_divide(sum(ae_total), sum(actual_total))").alias("weekly_level_wape_total")
    )
)

display(daily_total_capture_summary)
display(weekly_total_capture_summary)

# COMMAND ----------

# Create a confusion matrix for zero vs non-zero predictions
# This diagnostic shows how well the model handles the zero-inflation problem
# (when demand is truly zero vs when the model incorrectly predicts zero)
zero_diagnostics = (
    pred_final
    .select(
        # Pretzel zero-prediction analysis: 2x2 contingency table
        F.sum(((F.col("Sales of Pretzels") == 0) & (F.col("pred_pretzels") == 0)).cast("int")).alias("pretzel_actual0_pred0"),  # True negatives
        F.sum(((F.col("Sales of Pretzels") == 0) & (F.col("pred_pretzels") > 0)).cast("int")).alias("pretzel_actual0_predpos"),  # False positives
        F.sum(((F.col("Sales of Pretzels") > 0) & (F.col("pred_pretzels") == 0)).cast("int")).alias("pretzel_actualpos_pred0"),  # False negatives
        F.sum(((F.col("Sales of Pretzels") > 0) & (F.col("pred_pretzels") > 0)).cast("int")).alias("pretzel_actualpos_predpos"),  # True positives

        # Nugget zero-prediction analysis: identical structure to pretzels
        F.sum(((F.col("Sales of Pretzel Nuggets") == 0) & (F.col("pred_nuggets") == 0)).cast("int")).alias("nugget_actual0_pred0"),
        F.sum(((F.col("Sales of Pretzel Nuggets") == 0) & (F.col("pred_nuggets") > 0)).cast("int")).alias("nugget_actual0_predpos"),
        F.sum(((F.col("Sales of Pretzel Nuggets") > 0) & (F.col("pred_nuggets") == 0)).cast("int")).alias("nugget_actualpos_pred0"),
        F.sum(((F.col("Sales of Pretzel Nuggets") > 0) & (F.col("pred_nuggets") > 0)).cast("int")).alias("nugget_actualpos_predpos")
    )
)

display(zero_diagnostics)

# COMMAND ----------

# Compare the ML model's performance against a baseline forecasting method
# This shows whether the model actually adds value vs a simple benchmark
baseline_vs_model = (
    pred_final
    # Calculate totals (pretzels + nuggets) for actual, model prediction, and baseline prediction
    .withColumn("actual_total", F.col(ACT_P) + F.col(ACT_N))
    .withColumn("pred_total", F.col("pred_pretzels") + F.col("pred_nuggets"))
    .withColumn("baseline_total", F.col("baseline_p") + F.col("baseline_n"))
    # Calculate absolute error for baseline method
    .withColumn("baseline_ae_total", F.abs(F.col("baseline_total") - F.col("actual_total")))
    # Calculate absolute error for ML model
    .withColumn("model_ae_total", F.abs(F.col("pred_total") - F.col("actual_total")))
    # Flag rows where model outperforms baseline (lower error = better)
    .withColumn("model_beats_baseline", (F.col("model_ae_total") < F.col("baseline_ae_total")).cast("double"))
    .agg(
        # Calculate percentage of observations where model has lower error than baseline
        F.mean("model_beats_baseline").alias("pct_rows_model_beats_baseline"),
        # Calculate average improvement in absolute error (negative = model worse, positive = model better)
        F.mean(F.col("model_ae_total") - F.col("baseline_ae_total")).alias("mean_ae_change_vs_baseline")
    )
)

display(baseline_vs_model)

forecast_date,actual_pretzels,actual_nuggets,pred_pretzels,pred_nuggets,actual_total,pred_total,ae_p,ae_n,ae_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06,38140.0,115603.0,27672.244072030677,81434.38095944856,153743.0,109106.62503147924,10467.755927969323,34168.619040551435,44636.37496852076,0.2744561071832544,0.29556861881224045,0.29033110430081865
2026-04-07,34009.0,93796.0,28692.709491491292,79956.63621216344,127805.0,108649.34570365473,5316.290508508708,13839.363787836555,19155.654296345267,0.15632010669260218,0.1475474837715527,0.1498818848741854
2026-04-08,33929.0,95592.0,28965.947667315562,83997.07081308824,129521.0,112963.0184804038,4963.052332684438,11594.929186911759,16557.9815195962,0.14627758945693767,0.12129602045057912,0.1278401303232387
2026-04-09,35337.0,98670.0,30896.62268250561,91944.22068377904,134007.0,122840.84336628465,4440.377317494389,6725.779316220956,11166.156633715349,0.1256580161726912,0.0681643794083405,0.08332517430966553
2026-04-10,44379.0,128793.0,43485.67017935984,139456.26808074955,173172.0,182941.9382601094,893.3298206401596,10663.268080749549,9769.938260109397,0.020129561744071735,0.08279384811868307,0.056417540134140604
2026-04-11,66656.0,214809.0,63986.08086603703,219448.49577164542,281465.0,283434.57663768245,2669.9191339629688,4639.495771645423,1969.5766376824467,0.04005519584077906,0.021598237372016178,0.006997589887490263
2026-04-12,46985.0,148425.0,37476.562919531665,121471.63434137317,195410.0,158948.19726090482,9508.437080468335,26953.365658626833,36461.802739095176,0.20237175865634427,0.18159586093061703,0.18659128365536654
2026-04-13,26938.0,68054.0,29135.61156721333,88218.53673947002,94992.0,117354.14830668335,2197.611567213331,20164.536739470015,22362.148306683346,0.08158035367188844,0.2963020063401125,0.23541085887951982
2026-04-14,27112.0,65276.0,29498.973205327027,82840.19373014985,92388.0,112339.16693547688,2386.9732053270272,17564.193730149855,19951.166935476882,0.08804120704215945,0.2690758277184548,0.2159497655050102
2026-04-15,27163.0,69887.0,29635.294248968126,86281.27531952193,97050.0,115916.56956849006,2472.2942489681263,16394.275319521927,18866.569568490057,0.0910169807815089,0.23458261650266754,0.19440051075208714


forecast_week_start,actual_pretzels,actual_nuggets,pred_pretzels,pred_nuggets,actual_total,pred_total,ae_p,ae_n,ae_total,wape_pretzels,wape_nuggets,wape_total
2026-04-06T00:00:00.000Z,299435.0,895688.0,261175.8378782716,817708.7068622477,1195123.0,1078884.5447405193,38259.16212172841,77979.29313775233,116238.45525948075,0.12777117612078887,0.08706077689748254,0.09726066292714704
2026-04-13T00:00:00.000Z,265201.0,755702.0,264641.52515868115,824962.4471677521,1020903.0,1089603.9723264333,559.4748413188499,69260.44716775208,68700.97232643329,0.0021096256851175143,0.09165047487998189,0.06729431917276497


store_volume_segment,N,sum_ae_p,sum_ae_n,sum_ae_total,pretzel_volume,nugget_volume,total_volume,bias_pretzels,bias_nuggets,bias_total,wape_pretzels,wape_nuggets,wape_total
high,62608,136124.81568917842,295166.4516122469,369280.86661911535,328822.0,929796.0,1258618.0,-0.2597187242211402,0.04149299664612271,-0.21822572757501743,0.4139772146911655,0.3174529161367084,0.29340186348766295
low,62426,52007.556136273175,94664.98312763334,121667.35981958572,76623.0,205627.0,282250.0,-0.20775895893087062,-0.2181721977374393,-0.42593115666831,0.6787460179877214,0.46037233985630943,0.43106239085769965
medium,62608,86778.99280179125,194133.11103218622,235823.75690055508,159191.0,515967.0,675158.0,-0.15315305243405652,0.03678408688884252,-0.11636896554521396,0.5451249932583578,0.376251021930058,0.34928676976434414


daily_level_wape_pretzels,daily_level_wape_nuggets,daily_level_wape_total
0.1162184730992794,0.14422622190523357,0.13233308937359609


weekly_level_wape_pretzels,weekly_level_wape_nuggets,weekly_level_wape_total
0.06874984408193435,0.08916109477803842,0.0834554412204163


pretzel_actual0_pred0,pretzel_actual0_predpos,pretzel_actualpos_pred0,pretzel_actualpos_predpos,nugget_actual0_pred0,nugget_actual0_predpos,nugget_actualpos_pred0,nugget_actualpos_predpos
63680,21934,1400,100628,58630,9719,964,118329


pct_rows_model_beats_baseline,mean_ae_change_vs_baseline
0.4217925624327176,-0.29688991089811245
